# DLNM Pilot - Toronto Heat-Mortality Validation

Small-area heat-mortality vulnerability mapping for Canadian CMAs, structurally mirroring Gasparrini et al. 2022. DAs play the role of LSOAs; CMAs the role of LADs. This pilot uses simulated mortality with three latent vulnerability factors so recovery can be checked - everything else (geography, temperature, age demographics) is real Toronto 2021/2015-2019 data.


This notebook is the running pipeline for the small-area heat-mortality vulnerability map — built up in stages, validated end to end on city slivers before scaling. Real Daymet temperature → simulated mortality (known latent structure) → Stage 1 conditional-Poisson DLNM → reduce → Stage 2 pooling → DA downscale. Stage 1, reduce, and the first non-degenerate Stage 2 pool (3 CMAs: Toronto, Montréal, Vancouver, age 75–84) run here. Stage 3 downscale and the full-Toronto Stage 1 (Compute Canada) are gated on compute; real mortality is gated on data clearance.

Notation: 3 internal spline knots → 5 basis functions per margin. Cross-basis = 5 (temp) × 5 (lag) = 25 terms; reduced curve = 5 terms.

Contents

0. Restore from session
1. Polygon geography (Toronto)
2. Age demographics (Toronto)
3. Save milestone
4. Daymet temperature
5. Simulate mortality
6. Stage 1 DLNM
7. Reduce + Stage 2 (3-CMA pool)
8. Downscale + Figure 3 (+ Criterion 3)
9. Scaling—MTL + VAN to bronze
10. Axis reconciliation + per-DA downscale
11. Bottom-up rotation (final form) + diagnosis

## Packages

'dlnm' and 'mixmeta' are Gasparrini's; the rest support I/0 and geospatial work

In [ ]:
library(dlnm)
library(gnm)
library(mixmeta)
library(splines)
library(sf)
library(data.table)
library(daymetr)
library(ggplot2)
library(viridis)
library(exactextractr)
library(terra)
library(arrow)
library(lubridate)

## 3. Constants

Every value the pipeline reads at a call site sits here. Values that are already function arguments stay with their functions.

`cma_ref` replaces three separate DGUID vectors and the seed offsets that were typed inline at each `run_city` call. The offsets are historical, not derivable: Toronto, Montréal and Vancouver took their province prefix, and the three later cities took an arbitrary distinct integer once Montréal and Québec City were found to share prefix 24. Every banked θ depends on these values, so they are fixed, and the only requirement is that no two cities share one.

Calgary's GEONO is absent. Alberta has no DA-level Census Profile of its own — the files are regional and Alberta sits inside Prairies.
"""

In [ ]:
DRIVE        <- "/content/drive/MyDrive/thesis/dlnm-pilot"
EE_PROJECT   <- "gen-lang-client-0569405164"
EE_ASSET_DIR <- sprintf("projects/%s/assets", EE_PROJECT)

year_start  <- 2015
year_end    <- 2019
warm_months <- 5:9
study_dates <- seq.Date(as.Date(sprintf("%d-05-01", year_start)),
                        as.Date(sprintf("%d-09-30", year_end)), by = "day")
study_dates <- study_dates[lubridate::month(study_dates) %in% warm_months]

cma_ref <- data.table(
  cma         = c("Toronto","Montreal","Vancouver","Ottawa","Calgary","Quebec"),
  dguid       = c("2021S0503535","2021S0503462","2021S0503933",
                  "2021S0503505","2021S0503825","2021S0503421"),
  seed_offset = c(35L, 24L, 59L, 63L, 825L, 421L),
  geono       = c("006_Ontario","006_Quebec","006_BC_CB",
                  "006_Ontario;006_Quebec", NA_character_, "006_Quebec"),
  profile_sfx = c("Ontario","Quebec","BritishColumbia",
                  "Ontario;Quebec", NA_character_, "Quebec")
)
stopifnot(!any(duplicated(cma_ref$seed_offset)), !any(duplicated(cma_ref$dguid)))

annual_rates <- c(age_0_64 = 0.5, age_65_74 = 20, age_75_84 = 50, age_85p = 150)

L <- matrix(0, nrow = 17, ncol = 3)
L[1:5, 1]     <- c(0.9, 0.7, 0.6, -0.5, 0.7)
L[6:10, 2]    <- c(-0.8, 0.9, 0.7, 0.6, 0.7)
L[11:15, 3]   <- c(0.8, 0.7, 0.5, 0.4, 0.5)
L[16, c(1,2)] <- c(0.4, 0.4)
L[17, c(2,3)] <- c(0.3, 0.5)
true_loadings <- L

lag_weights <- dgamma(0:21, shape = 2, rate = 1.2)
lag_weights <- lag_weights / sum(lag_weights)

age_ids  <- c(10,11,12,14,15,16,17,18,19,20,21,22,23, 25,26, 27,28, 29)
band_map <- data.table(
  CHARACTERISTIC_ID = age_ids,
  band = c(rep("age_0_64",13), rep("age_65_74",2), rep("age_75_84",2), "age_85p")
)

cdn_2011_std <- c(age_0_64 = 0.8210, age_65_74 = 0.0930,
                  age_75_84 = 0.0580, age_85p = 0.0280)
stopifnot(abs(sum(cdn_2011_std) - 1) < 1e-6)

cat("study_dates:", length(study_dates), "\n")
cat("cma_ref rows:", nrow(cma_ref), "  Calgary geono:", cma_ref[cma == "Calgary"]$geono, "\n")
cat("L:", paste(dim(L), collapse = " x "), "  lag_weights sum:", sum(lag_weights), "\n")
cat("age_ids:", length(age_ids), "  band_map rows:", nrow(band_map), "\n")

# 4. Functions

Every function the pipeline calls is defined here and nowhere else. A function defined inside a stage cell is invisible to `source()` and survives only in the session that made it, which is how five functions once became live in memory and absent from `fns.R`.

`fns.R` is written from this section by §4.19. It is not edited by hand. Open flag (2026-08-21): fns.R currently defines fit_stage1 three times (last wins) — resolve in the overhaul, not by hand-editing.

## Restore from previous session

Two tarballs restore the session: r_library.tar.gz (package cache → /content/site-library, skips the 10-min reinstall) and the latest saves_pilot snapshot (data + functions). load() restores objects but not library() calls — re-attach packages explicitly, or every fread/crossbasis call throws "could not find function".

Everything needed to resume lives in Drive at '/content/drive/MyDrive/thesis/dlnm-pilot'. What each file is:

- r_library.tar.gz - compiled R package cache. Unpacks to /content/site-library; skips the ~2hr reinstall.
- saves_pilot_2026-06-03.tar.gz - DGP machinery + light inputs (simulate_counts, base_log_rr, lag_weights, Z, truth_factors, da_age, true_loadings, daymet_csv path). Carries stale qaic/reduce_fit — re-pull the fixed versions from the EOD tarball after load(). Does not hold heavy derived objects (sim, temp_mat, sliver, reduce); those regenerate at seed 42.
- saves_mtlvan_clean.tar.gz - Montréal + Vancouver geography (DA lists + polygons). 4.68 MB.
- saves_eod_2026-06-25.tar.gz - v1 provenance, no longer loaded by §0 (2026-08-21): everything it was read for — cma_age_data_mtlvan, mtl/van substrates — is duplicated inside 07-22. Also holds mtl/van substrates and the v1 reduced curves; those are superseded. It contains fn_fit_stage1/fn_qaic/fn_reduce_fit; those are not read by §0's named readRDS calls, but any loop-assign restore will load them as live functions — rm on sight (confirmed 2026-08-05). Functions come from fns.R.
- saves_eod_2026-07-22.tar.gz - current resume point, v2. red5_v2, res5_v2, Z5_v2, ids5_v2, diag5_v2, stage2_k5_v2, cma_predictors, new_age_ottqc. The Ottawa/Québec age tables exist nowhere else — regenerating them costs 1.3 GB of profile downloads and a grep over 14 GB.
- mtl_sim_2026-06-24.rds / mtl_da_long_2026-06-24.rds / mmt_da_mtl_2026-06-24.rds - loose dated files, v1 additive DGP. Superseded; do not restore into a v2 session. Regenerable at seed 42 from §9.6 if ever needed.
- montreal_daymet_2015_2019.csv / vancouver_daymet_2015_2019.csv - Daymet warm-season temperature per CMA, ~5.0M / 2.7M rows.
- toronto_daymet_2015_2019.csv - Daymet warm-season temperature, 5.9M rows. Read by §5.4.
- ottawa_daymet_2015_2019.csv / calgary_daymet_2015_2019.csv / quebec_daymet_2015_2019.csv - 1,564,426 / 1,451,971 / 1,007,506 rows.
- toronto_da_shp.zip / montreal_da_shp.zip / vancouver_da_shp.zip / ottawa_da_shp.zip / calgary_da_shp.zip / quebec_da_shp.zip - DA shapefile bundles; the Earth Engine reduce-by-region assets.

Reload order:
(1) pull library cache + re-attach full package stack
(2) load saves_pilot_2026-06-03 — base_log_rr, lag_weights, L, da_age, annual_rates, daymet_csv. Also carries stale functions and the v1 truth_factors imposter; step 3 overwrites the functions, the imposter is rm'd on sight.
(3) source(fns.R) — must be after step 2. All 17 pipeline functions live here.
(4) readRDS saves_eod_2026-07-22 by name — reduced curves, Z, ids, diag, stage2 pool, ottawa/québec age tables, mtl/van substrates, cma_age_data_mtlvan. Named reads only; a loop-assign would make the tarball's fn_*.rds live functions.
(5) run constants cell

Then the body runs top-to-bottom: §5.4 rebuilds Toronto temp_mat, §5.5 fires the Toronto sim, §6.4 fits a sliver, §6.5 builds the VAN sim, §7.3 reduces, §7.4 pools.

In [ ]:
DRIVE <- "/content/drive/MyDrive/thesis/dlnm-pilot"

system(sprintf("tar -xzf '%s/r_library.tar.gz' -C /content", DRIVE))
.libPaths(c("/content/site-library", .libPaths()))
suppressPackageStartupMessages({
  library(dlnm); library(gnm); library(mixmeta); library(splines)
  library(sf); library(data.table); library(ggplot2); library(viridis); library(lubridate)
})

system(sprintf("tar -xzf '%s/saves_pilot_2026-06-03.tar.gz' -C /content", DRIVE))
rdata <- list.files("/content", pattern = "pilot_session\\.RData$", recursive = TRUE, full.names = TRUE)
stopifnot(length(rdata) == 1)
pre <- ls()
load(rdata)
loaded <- setdiff(ls(), c(pre, "pre", "rdata"))

source(file.path(DRIVE, "fns.R"))
if (exists("truth_factors")) rm(truth_factors)

system(sprintf("tar -xzf '%s/saves_eod_2026-07-22.tar.gz' -C /content", DRIVE))
EOD <- "/content/saves_eod"
red5           <- readRDS(file.path(EOD, "red5_v2.rds"))
res5           <- readRDS(file.path(EOD, "res5_v2.rds"))
Z_list         <- readRDS(file.path(EOD, "Z5_v2.rds"))
ids5           <- readRDS(file.path(EOD, "ids5_v2.rds"))
diag5          <- readRDS(file.path(EOD, "diag5_v2.rds"))
cma_predictors <- readRDS(file.path(EOD, "cma_predictors.rds"))
new_age        <- readRDS(file.path(EOD, "new_age_ottqc.rds"))
stage2_k5      <- readRDS(file.path(EOD, "stage2_k5_v2.rds"))$fit
mtl            <- readRDS(file.path(EOD, "mtl_substrate.rds"))
van            <- readRDS(file.path(EOD, "van_substrate.rds"))
cma_age_data_mtlvan <- readRDS(file.path(EOD, "cma_age_data_mtlvan.rds"))

need <- c("read_daymet","build_crossbasis","make_strata_A","make_strata_B","qaic",
          "fit_city_sliver","build_city_sim_substrate_v2","simulate_counts",
          "fit_stage1","reduce_fit","fit_stage2","fit_da_pca","predict_da_theta",
          "compute_da_mmt","monte_carlo_ci","standardize_da_rate","save_to_drive",
          "base_log_rr","make_choropleth")
fns_txt <- readLines(file.path(DRIVE, "fns.R"))
audit <- data.table(fn = need,
  in_env  = sapply(need, function(f) exists(f) && is.function(get(f))),
  in_file = sapply(need, function(f) any(grepl(sprintf("^%s <- function", f), fns_txt))))
audit[, landmine := in_env & !in_file]

cities <- c("Toronto","Montreal","Vancouver","Ottawa","Quebec")

cat("landmines:", sum(audit$landmine), "\n")
cat("tar 07-22 rds:", length(list.files(EOD, pattern = "\\.rds$")), "\n")
cat("simulate_counts exp:", any(grepl("exp\\(", deparse(body(simulate_counts)))), "\n")
cat("red5 names:", names(red5), "\n")
cat("session objects:", length(loaded), "| truth_factors gone:", !exists("truth_factors"), "\n")
cat("da_age rows:", nrow(da_age), "| L:", paste(dim(L), collapse = " x "), "\n")

"""landmines: 0
tar 07-22 rds: 27
simulate_counts exp: TRUE
red5 names: Toronto Montreal Vancouver Ottawa Quebec
session objects: 47 | truth_factors gone: TRUE
da_age rows: 7694 | L: 17 x 3
"""

## 0.1 Save to Drive

Bundles named objects to /content/saves_eod, tarballs, copies to Drive — one call, no hand-typed paths. Restore (§0) reads the same tarball shape. Call: save_to_drive(list(mtl_sim = mtl_sim, mtl_da_long = mtl_da_long, mmt_da = mmt_mtl)).

In [ ]:
save_to_drive <- function(objs, tag = format(Sys.Date(), "%Y-%m-%d"), drive = DRIVE) {
  dir.create("/content/saves_eod", showWarnings = FALSE)
  for (nm in names(objs)) saveRDS(objs[[nm]], sprintf("/content/saves_eod/%s.rds", nm))
  tarball <- sprintf("saves_eod_%s.tar.gz", tag)
  system(sprintf("cd /content && tar -czf %s saves_eod/ && cp %s '%s/'", tarball, tarball, drive))
  cat("saved", length(objs), "objects ->", file.path(drive, tarball), "\n")
}

# 1. Polygon Geography

Direct from StatCan. The cancensus API failed on quota for DA-level pulls, so we bypassed it and went straight to the boundary file + relationship file

## 1.1 Download national DA shapefile

Statcan publishes the 2021 cartographic boundary file for all Canadian DAs at a permanent url.

In [ ]:
dir.create("/content/statcan", showWarnings = FALSE)

da_zip_url  <- "https://www12.statcan.gc.ca/census-recensement/2021/geo/sip-pis/boundary-limites/files-fichiers/lda_000b21a_e.zip"
da_zip_path <- "/content/statcan/da_boundaries.zip"
da_dir      <- "/content/statcan/da_extracted"

if (!file.exists(da_zip_path)) {
  download.file(da_zip_url, da_zip_path, mode = "wb")
}
if (!dir.exists(da_dir)) {
  dir.create(da_dir)
  unzip(da_zip_path, exdir = da_dir)
}
list.files(da_dir)

Result: 4 shapefile pieces (.shp/.shx/.dbf/.prj) plus an .xml metadata file.

In [ ]:
shp_path <- "/content/statcan/da_extracted/lda_000b21a_e.shp"
da_all <- sf::st_read(shp_path, quiet = TRUE)
cat("Rows:", nrow(da_all), "  Cols:", ncol(da_all), "\n")
cat("CRS:", sf::st_crs(da_all)$input, "\n")
print(colnames(da_all))
head(sf::st_drop_geometry(da_all), 3)

Result: 57,932 DAs nationwide. Columns: `DAUID`, `DGUID`, `LANDAREA`, `PRUID`, `geometry`. CRS = NAD83 / Statistics Canada Lambert. **no `CMAUID` column.** The shapefile only knows province. We need the Dissemination Geographics Relationship File to map DA -> CMA

## 1.2 Download Dissemination Geographies Relationship File (DGRF)

DGRF maps every dissemination block to all higher geographic levels via DGUIDs. DGUID = `<year><type-prefix><code>` — e.g. `2021S0503535` = 2021 statistical CMA 535 = Toronto. Globally unique across the whole geography hierachy

In [ ]:
dgrf_zip_url  <- "https://www12.statcan.gc.ca/census-recensement/2021/geo/sip-pis/dguid-idugd/files-fichiers/2021_98260004.zip"
dgrf_zip_path <- "/content/statcan/dgrf.zip"
dgrf_dir      <- "/content/statcan/dgrf_extracted"

if (!file.exists(dgrf_zip_path)) {
  download.file(dgrf_zip_url, dgrf_zip_path, mode = "wb")
}
if (!dir.exists(dgrf_dir)) {
  dir.create(dgrf_dir)
  unzip(dgrf_zip_path, exdir = dgrf_dir)
}

dgrf <- data.table::fread("/content/statcan/dgrf_extracted/2021_98260004.csv")
cat("Rows (DBs):", nrow(dgrf), " Cols:", ncol(dgrf), "\n")


Result: 498,786 dissemination blocks x 16 columns. File is at the DB level, not DA. Each DA appears multiple times.

## 1.3 Filter DGRF to Toronto CMA, extract unique DA list

Toronto CMA DGUID = `2021S0503535`. After filer, deduplicate to DA level

In [ ]:
toronto_cma_dguid <- "2021S0503535"

toronto_db_rows   <- dgrf[CMADGUID_RMRIDUGD == toronto_cma_dguid]
toronto_da_dguids <- unique(toronto_db_rows$DADGUID_ADIDUGD)
toronto_dauids    <- substr(toronto_da_dguids, 10, 17)

cat("DBs:", nrow(toronto_db_rows), "  Unique DAs:", length(toronto_dauids), "\n")
cat("Unique province prefixes:", paste(unique(substr(toronto_dauids, 1, 2)), collapse = ", "), "\n")

Result: 34,206 DBs -> 7,716 unique DAs. All begin with `35` (Ontario). Avg ~4.4 DBs per DA, matches design.

## 1.4 Join DAUIDs to national shapefile -> Toronto polygons

`%in` filer against the 7,716-element vector

In [ ]:
da_all$DAUID <- as.character(da_all$DAUID)
toronto_da   <- da_all[da_all$DAUID %in% toronto_dauids, ]

cat("Rows after filter:", nrow(toronto_da), "\n")
cat("Missing:", length(setdiff(toronto_dauids, toronto_da$DAUID)), "\n")
cat("Land area:", round(sum(toronto_da$LANDAREA), 1), "km²\n")


Result: 7,716 polygons, 0 missing, total area 5902.7 km². Matches StatCan's published Toronto CMA area to within 0.05%

# 2. Age Demographics

Census Profile (catalogue 98-401-X2021006). The Ontario subset is the right grain

## 2.1 Download Ontario Profile

740 MB zipped CSV. Unzips to 8.4 GB long-format CSV (~50M rows, one row per (geography, characteristic)).

In [ ]:
profile_url  <- "https://www12.statcan.gc.ca/census-recensement/2021/dp-pd/prof/details/download-telecharger/comp/GetFile.cfm?Lang=E&FILETYPE=CSV&GEONO=006_Ontario"
profile_zip  <- "/content/statcan/profile_ontario.zip"
profile_dir  <- "/content/statcan/profile_extracted"

if (!file.exists(profile_zip)) {
  options(timeout = 1800)
  download.file(profile_url, profile_zip, mode = "wb")
}
if (!dir.exists(profile_dir)) {
  dir.create(profile_dir)
  unzip(profile_zip, exdir = profile_dir)
}
profile_csv <- "/content/statcan/profile_extracted/98-401-X2021006_English_CSV_data_Ontario.csv"

Result: ~720 MB zip -> 8.4 GV csv. The url ends in `&FILETYPE=CSV` but the response is actually a zipped CSV

## 2.2 Filter to Toronto DAs via bash grep

Reading 8.4 GB into R then filtering wastes memory. Bash `grep` streams the file at the filesystem level; never holds more than a buffer in RAM. Write the 7,716 DAUIDs to a patterns file, then `grep -F -f patterns.txt` extracts only matching lines

In [ ]:
patterns_path     <- "/content/statcan/toronto_patterns.txt"
profile_toronto_csv <- "/content/statcan/profile_toronto.csv"

writeLines(paste0('"', toronto_dauids, '"'), patterns_path)

if (!file.exists(profile_toronto_csv)) {
  system(sprintf('head -1 "%s" > "%s"', profile_csv, profile_toronto_csv))
  system(sprintf('grep -F -f "%s" "%s" >> "%s"',
                 patterns_path, profile_csv, profile_toronto_csv))
}

cat("Filtered size:", round(file.info(profile_toronto_csv)$size / 1024^2, 1), "MB\n")


Result: 8.4 GB -> ~3 GB. About 20M rows survive (7,716 DAs x ~2610 characteristics each)

## 2.3 Locate the 18 age-bin CHARACTERISTIC_IDs in the meta file

CensusMapper labels age vectors as 'v_CA21_X'. StatCan Profile labels them with integer CHARACTERISTIC_IDs—different numbering. The meta file (`*_English_meta.txt`) is the dictionary mapping ID -> name. The "Total - Age" block starts at ID 8

| Band      | IDs                                          | Bins                        |
|-----------|----------------------------------------------|-----------------------------|
| 0–64      | 10–12, 14–23                                 | 0-4 through 60-64           |
| 65–74     | 25, 26                                       | 65-69, 70-74                |
| 75–84     | 27, 28                                       | 75-79, 80-84                |
| 85+       | 29                                           | 85 and over

In [ ]:
age_ids <- list(
  age_0_64  = c(10, 11, 12, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23),
  age_65_74 = c(25, 26),
  age_75_84 = c(27, 28),
  age_85p   = c(29)
)
all_age_ids <- unlist(age_ids, use.names = FALSE)

## 2.4 Read, filter, aggregate, pivot

`data.table::fread` with `select=` skips columns at read time—only loads 4 of 23 cols. Then filter to age IDs, aggregate to 4 bands, pivot wide

In [ ]:
profile_toronto <- data.table::fread(
  profile_toronto_csv,
  select = c("DGUID", "ALT_GEO_CODE", "CHARACTERISTIC_ID", "C1_COUNT_TOTAL")
)
profile_ages <- profile_toronto[CHARACTERISTIC_ID %in% all_age_ids]
rm(profile_toronto); gc()

band_map <- data.table::data.table(
  CHARACTERISTIC_ID = c(10, 11, 12, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23,
                        25, 26, 27, 28, 29),
  band = c(rep("age_0_64", 13), rep("age_65_74", 2),
           rep("age_75_84", 2), rep("age_85p", 1))
)
profile_ages <- merge(profile_ages, band_map, by = "CHARACTERISTIC_ID")
profile_ages[is.na(C1_COUNT_TOTAL), C1_COUNT_TOTAL := 0]

da_age <- profile_ages[, .(pop = sum(C1_COUNT_TOTAL)),
                       by = .(ALT_GEO_CODE, band)]
da_age_wide <- data.table::dcast(da_age, ALT_GEO_CODE ~ band, value.var = "pop")
da_age_wide[, total := age_0_64 + age_65_74 + age_75_84 + age_85p]

cat("DAs:", nrow(da_age_wide), "  Total pop:", sum(da_age_wide$total), "\n")


Result: 7,716 DAs. Total population 6,200, 815 (matches StatCan's published Toronto CMA total to within 0.02%)

Band percentages: 0-64 = 83.8%, 65-74 = 9.2%, 75-84 = 4.9%, 85+ = 2.1%

## 2.5 Join age bands onto polygons

Final artifact is one `sf` object with polygon geometry + 4 age band columns per DA

In [ ]:
da_age_wide[, ALT_GEO_CODE := as.character(ALT_GEO_CODE)]
toronto_da_full <- merge(toronto_da, da_age_wide,
                         by.x = "DAUID", by.y = "ALT_GEO_CODE",
                         all.x = TRUE)
cat("Final rows:", nrow(toronto_da_full),
    "  DAs with no age data:", sum(is.na(toronto_da_full$age_0_64)), "\n")

Result: 7,716 DAs with geometry + 4 age bands, zero missing.

# 3. Save Milestone

Bundle the .rds files into a tarball, download to Drive. Without this, Colab disconnects and I redo 4 hours of work.

In [ ]:
dir.create("/content/saves", showWarnings = FALSE)
saveRDS(toronto_da_full, "/content/saves/toronto_da_full.rds")
saveRDS(toronto_dauids,  "/content/saves/toronto_dauids.rds")
saveRDS(da_age_wide,     "/content/saves/da_age_wide.rds")

today <- format(Sys.Date(), "%Y-%m-%d")
tarball <- sprintf("/content/saves_%s.tar.gz", today)
system(sprintf("cd /content && tar -czf saves_%s.tar.gz saves/", today))
cat("Saved:", tarball, "—",
    round(file.info(tarball)$size / 1024^2, 2), "MB\n")

Tarball at `/content/saves_YYYY-MM-DD.tar.gz`. Download manually to Drive.

To restore next session: upload tarball -> `tar -xzf` -> `readRDS()` the three files

# 4. Functions

Every function the pipeline calls, in execution order, one per cell. `fns.R` is this section concatenated — it is exported, not edited.

Two definitions change here. `simulate_counts` gains the closing brace its loop was missing. `monte_carlo_ci` now takes the fit object rather than the wrapper, matching `predict_da_theta`.
"""

# 4. Daymet Temperature via Earth Engine

`daymetr` 1.7.1's THREDDS endpoint was deprecated when ORNL migrated Daymet V4 to AWS cloud in 2024. Per ORNL staff, current access methods are AppEEARS, Google Earth Engine, or OPeNDAP DAP4. Earth Engine is the path of least resistance: polygon-native, server-side reduction, returns a CSV.

## 4.1 Export DA polygons as Earth Engine asset

EE doesn't accept GeoJSON on upload; needs a zipped shapefile bundle (.shp/.shx/.dbf/.prj). Write the shapefile from `toronto_da_full`, zip it, upload via the EE Code Editor as `projects/<project-id>/assets/toronto_da`.

In [ ]:
library(sf)
out_dir <- "/content/toronto_da_shp"
dir.create(out_dir, showWarnings = FALSE)

st_write(toronto_da_full["DAUID"],
         file.path(out_dir, "toronto_da.shp"),
         delete_dsn = TRUE, quiet = TRUE)

zip_path <- sprintf("%s/toronto_da_shp.zip", DRIVE)
zip(zipfile = zip_path, files = list.files(out_dir, full.names = TRUE), flags = "-j")

n_entries <- length(unzip(zip_path, list = TRUE)$Name)
cat("Shapefile bundle:", round(file.size(zip_path)/1e6, 2), "MB  entries:", n_entries, "\n")

"""Result: 5.16 MB zipped, 4 entries (.shp/.shx/.dbf/.prj). The `-j` flag flattens paths so EE sees a clean bundle; the older setwd+zip form silently wrote empty zips. Uploaded to EE via Code Editor sidebar. Asset ID: `projects/gen-lang-client-0569405164/assets/toronto_da`.

## 4.2 Server-side reduce in Earth Engine (Python)

Daymet V4 is in EE's catalog as `NASA/ORNL/DAYMET_V4`. Filter to May-Sep 2015-2019, reduce by region across the 7,716 DA polygons, export to Drive as CSV. The reduce happens on Google's cluster — 5.9M individual mean calculations done server-side. Run in a separate Python notebook (`Daymet_Pull.ipynb`) to avoid mixing R and Python kernels mid-session.

In [ ]:
```python
import ee
ee.Authenticate()
ee.Initialize(project='gen-lang-client-0569405164')

da_fc  = ee.FeatureCollection('projects/gen-lang-client-0569405164/assets/toronto_da')
daymet = (ee.ImageCollection('NASA/ORNL/DAYMET_V4')
          .filterDate('2015-05-01', '2019-10-01')
          .filterBounds(da_fc.geometry().bounds())
          .select(['tmax', 'tmin']))

def keep_warm(img):
    m = ee.Date(img.get('system:time_start')).get('month')
    return img.set('keep', m.gte(5).And(m.lte(9)))

daymet_warm = daymet.map(keep_warm).filter(ee.Filter.eq('keep', 1))

def reduce_one_day(img):
    date_str = ee.Date(img.get('system:time_start')).format('YYYY-MM-dd')
    reduced  = img.reduceRegions(collection=da_fc,
                                 reducer=ee.Reducer.mean(),
                                 scale=1000, tileScale=4)
    return reduced.map(lambda f: f.set('date', date_str))

all_reductions = daymet_warm.map(reduce_one_day).flatten()

task = ee.batch.Export.table.toDrive(
    collection=all_reductions,
    description='toronto_daymet_2015_2019',
    folder='thesis/dlnm-pilot',
    fileNamePrefix='toronto_daymet_2015_2019',
    fileFormat='CSV',
    selectors=['DAUID', 'date', 'tmax', 'tmin'])
task.start()
```

Result: 54 min export, 5,902,740 rows, 334.7 MB CSV. tmax mean 24.2°C, tmin mean 13.4°C. July 2018 mean tmax 28.3°C matches actual Toronto 2018 (famously hot summer). 9,180 NAs (0.15%) — DAs along harbor/marsh edges where the Daymet 1km grid mostly intersects water.

In [ ]:
daymet_csv <- "/content/drive/MyDrive/thesis/dlnm-pilot/toronto_daymet_2015_2019.csv"
da_daily_temp <- fread(daymet_csv)
da_daily_temp[, date := as.Date(date)]
da_daily_temp[, tmean_C := (tmax + tmin) / 2]

cat("Rows:", nrow(da_daily_temp), "\n")
cat("Date range:", as.character(range(da_daily_temp$date)), "\n")
summary(da_daily_temp[, .(tmax, tmin, tmean_C)])

# 5. Simulate Mortality

DGP: three latent vulnerability factors per DA. 17 observed variables generated via a known loading matrix L (17×3) plus N(0, 0.3) noise. Truth-loadings preserved for Criterion 3 (PCA recovery check). Per-DA baseline rates by age band, modulated by a U-shaped temperature response curve centered at the DA's own 80th percentile MMT. Effect distributed across lags 0–21 via Gamma(2, 1.2). Poisson draws on the final rate.

## 5.1 Filter zero-pop DAs, sample latent factors, build observed Z

22 DAs in Toronto have population 0 — parks, water bodies, empty industrial. Filter them out before they propagate into the simulation. Then sample 3 standard-normal factors per remaining DA, design a 17×3 loading matrix with primary + cross-loadings, multiply through, add Gaussian noise.

In [ ]:
set.seed(42)

da_age <- as.data.table(da_age_wide)
da_age <- da_age[total > 0]
cat("DAs after pop>0 filter:", nrow(da_age), "/", nrow(da_age_wide), "\n")

n_da <- nrow(da_age)
da_age[, da_idx := .I]

truth_factors <- data.table(
  DAUID  = da_age$ALT_GEO_CODE,
  da_idx = da_age$da_idx,
  F1     = rnorm(n_da),
  F2     = rnorm(n_da),
  F3     = rnorm(n_da)
)

L <- matrix(0, nrow = 17, ncol = 3)
L[1:5, 1]   <- c(0.9, 0.7, 0.6, -0.5, 0.7)   # F1: elderly/isolated/hot
L[6:10, 2]  <- c(-0.8, 0.9, 0.7, 0.6, 0.7)   # F2: deprived/bad housing
L[11:15, 3] <- c(0.8, 0.7, 0.5, 0.4, 0.5)    # F3: environmental
L[16, c(1,2)] <- c(0.4, 0.4)
L[17, c(2,3)] <- c(0.3, 0.5)
true_loadings <- L

F_mat <- as.matrix(truth_factors[, .(F1, F2, F3)])
Z <- F_mat %*% t(L) + matrix(rnorm(n_da * 17, sd = 0.3), nrow = n_da)
colnames(Z) <- paste0("vuln", sprintf("%02d", 1:17))

cat("Z dim:", dim(Z), "\n")

SyntaxError: invalid syntax (2253468426.py, line 3)

Result: 7,694 DAs after the pop>0 filter (22 dropped). Z is 7,694 × 17 with three known latent factors baked in. (Trimmed to 7,682 in §5.4 once temperature reveals 12 water-only DAs.)

## 5.2 Baseline rates + DA-day-age long table

Annual mortality rates per 1,000 by age band (StatCan crude rates, rough):
- 0–64: 0.5
- 65–74: 20
- 75–84: 50
- 85+: 150

Build a long-format table (DA × date × age_band), attach population, compute baseline daily lambda₀ = pop × rate / 1000 / 365.

In [ ]:
annual_rates <- c(age_0_64 = 0.5, age_65_74 = 20, age_75_84 = 50, age_85p = 150)

da_long <- CJ(da_idx = 1:n_da, date = study_dates, age_band = names(annual_rates))
setkey(da_long, da_idx, date, age_band)

pop_long <- melt(da_age[, .(da_idx, age_0_64, age_65_74, age_75_84, age_85p)],
                 id.vars = "da_idx", variable.name = "age_band", value.name = "pop")
pop_long[, age_band := as.character(age_band)]
da_long <- pop_long[da_long, on = c("da_idx", "age_band")]

da_long[, annual_rate := annual_rates[age_band]]
da_long[, lambda0     := pop * annual_rate / 1000 / 365]

cat("da_long rows:", nrow(da_long), " (expect", n_da * 765 * 4, ")\n")
cat("lambda0 range:", range(da_long$lambda0), "\n")

Result: 23,543,640 rows (7,694 × 765 × 4). lambda₀ range [0, 0.179] — max is 85+ in the largest DA, ~150/1000/yr × ~440 people / 365 ≈ 0.18 deaths/day.

## 5.3 DGP function — temperature-modulated counts

Takes a wide temperature matrix (n_da × n_days), applies a U-shaped log-RR around each DA's MMT, modulates by F1/F2/F3, convolves with the lag kernel, scales lambda₀, draws Poisson counts. Returns the long table with `n_deaths`.

Modulation is multiplicative: `exp(0.4·F1·heat + 0.3·F2·cold + 0.2·F3)`. The additive `(1 + …)` form goes negative when a factor offset is large, inverting the U-curve on part of a city and breaking Stage 1; `exp(·)` stays positive. The recovery truth target is therefore `exp(0.4·F1 + 0.2·F3)`.

In [ ]:
base_log_rr <- function(temp, mmt) {
  ifelse(temp > mmt,
         0.03 * (temp - mmt)^2,
         0.01 * (temp - mmt)^2)
}

lag_weights <- dgamma(0:21, shape = 2, rate = 1.2)
lag_weights <- lag_weights / sum(lag_weights)

simulate_counts <- function(temp_mat, da_long, truth_factors, mmt_da, seed = 42) {
  set.seed(seed)
  n_da   <- nrow(temp_mat)
  n_days <- ncol(temp_mat)

  F1 <- truth_factors$F1; F2 <- truth_factors$F2; F3 <- truth_factors$F3
  log_rr_mat <- matrix(0, n_da, n_days)
  for (j in 1:n_days) {
    T_j     <- temp_mat[, j]
    base_j  <- base_log_rr(T_j, mmt_da)
    heat_ind <- pmax(T_j - mmt_da, 0) > 0
    cold_ind <- pmax(mmt_da - T_j, 0) > 0
    log_rr_mat[, j] <- base_j * exp(0.4*F1*heat_ind + 0.3*F2*cold_ind + 0.2*F3)

  log_rr_lagged <- matrix(0, n_da, n_days)
  for (i in 1:n_da) {
    log_rr_lagged[i, ] <- stats::filter(log_rr_mat[i, ], lag_weights, sides = 1)
  }
  log_rr_lagged[is.na(log_rr_lagged)] <- 0

  date_lookup <- data.table(date = study_dates, date_idx = seq_along(study_dates))
  da_long2 <- merge(da_long, date_lookup, by = "date", all.x = TRUE)
  setkey(da_long2, da_idx, date_idx, age_band)

  da_long2[, log_rr   := log_rr_lagged[cbind(da_idx, date_idx)]]
  da_long2[, lambda   := lambda0 * exp(log_rr)]
  da_long2[, n_deaths := rpois(.N, lambda)]

  return(da_long2)
}

cat("simulate_counts() defined. Fires at run-time after temp merge.\n")

## 5.4 Daymet → temperature matrix, drop water-only DAs

Read the 5.9M-row Daymet CSV, pivot long→wide into a DA × day matrix, align row order to `truth_factors` via `match()` (if rows don't align, every death bolts to the wrong DA's weather — silent garbage). Twelve DAs are NA on all 765 days — water-only polygons. Drop them from `temp_mat`, `truth_factors`, `Z` in lockstep so alignment holds. 7,694 → 7,682.

In [ ]:
da_daily_temp <- fread(daymet_csv, select = c("DAUID","date","tmax","tmin"))
da_daily_temp[, DAUID := as.character(DAUID)]
da_daily_temp[, date  := as.Date(date)]
da_daily_temp[, tmean_C := (tmax + tmin) / 2]

temp_wide <- dcast(da_daily_temp, DAUID ~ date, value.var = "tmean_C")
temp_wide <- temp_wide[match(truth_factors$DAUID, temp_wide$DAUID), ]
temp_mat  <- as.matrix(temp_wide[, -1])
rownames(temp_mat) <- temp_wide$DAUID

dead_rows <- which(rowSums(is.na(temp_mat)) == ncol(temp_mat))
keep      <- setdiff(seq_len(nrow(temp_mat)), dead_rows)

temp_mat      <- temp_mat[keep, ]
truth_factors <- truth_factors[keep, ]
Z             <- Z[keep, ]
n_da          <- length(keep)
truth_factors[, da_idx := .I]

cat("Dropped water-only DAs:", length(dead_rows), "\n")
cat("temp_mat:", paste(dim(temp_mat), collapse=" x "),
    " alignment:", all(rownames(temp_mat) == truth_factors$DAUID),
    " NA:", sum(is.na(temp_mat)), "\n")
cat("n_da:", n_da, "\n")

Result: 12 water-only DAs dropped (each NA on all 765 days — 9,180 / 12 = 765 exactly). temp_mat is 7,682 × 765, alignment TRUE, zero NA, range 2.2–30.1°C — sane Toronto warm-season. n_da = 7,682, the working DA count for everything downstream.

## 5.5 Rebuild da_long on 7,682, fire the simulation

`da_long` was built on 7,694 — rebuild it off the trimmed set so `da_idx` matches `temp_mat` rows. Then fire `simulate_counts`: U-shaped log-RR around each DA's 80th-percentile MMT, modulated by the latent factors, lag-convolved, Poisson-drawn. First time mortality exists.

In [ ]:
da_age_trim <- da_age[match(truth_factors$DAUID, da_age$ALT_GEO_CODE)]
da_age_trim[, da_idx := .I]

da_long <- CJ(da_idx = 1:n_da, date = study_dates, age_band = names(annual_rates))
pop_long <- melt(da_age_trim[, .(da_idx, age_0_64, age_65_74, age_75_84, age_85p)],
                 id.vars = "da_idx", variable.name = "age_band", value.name = "pop")
pop_long[, age_band := as.character(age_band)]
da_long <- pop_long[da_long, on = c("da_idx","age_band")]
da_long[, annual_rate := annual_rates[age_band]]
da_long[, lambda0 := pop * annual_rate / 1000 / 365]

mmt_da <- apply(temp_mat, 1, function(x) quantile(x, 0.80, na.rm=TRUE))
sim <- simulate_counts(temp_mat, da_long, truth_factors, mmt_da, seed = 42)

cat("da_long rows:", nrow(da_long), " NA pop:", sum(is.na(da_long$pop)), "\n")
cat("total deaths:", sum(sim$n_deaths), " NA:", sum(is.na(sim$n_deaths)), "\n")
print(sim[, .(deaths = sum(n_deaths)), by = age_band])

Result: da_long 23,506,920 rows (7,682 × 765 × 4), zero NA pop. 189,137 deaths over 5 warm-seasons. Deaths climb by age band: 10,233 → 43,381 → 58,313 → 77,210 (0-64 → 85+). 85+ has the fewest people but the most deaths — the steep old-age mortality slope reproduced correctly. 99.2% of cells zero (tiny lambdas) — which is exactly why Stage 1 needs conditional Poisson. Total runs ~2× real Toronto (~45-50k/yr all-cause): expected, not calibrated to real totals, calibrated to recover latent structure.

# 6. Stage 1 DLNM

Per (CMA, age) conditional Poisson DLNM via `gnm::gnm(... eliminate = strata)`. Cross-basis per Gasparrini 2022: quadratic B-spline on temperature (3 internal knots at 10/75/90th percentiles → 5 terms), natural cubic spline on lag (3 log-spaced knots over 0–21 days → 5 terms). Cross-basis is 5 × 5 = 25 columns. Both strata variants fitted, qAIC pick.

Full Toronto warm-season (all 7,682 DAs) is a 4–8h fit that exceeds Colab's idle window — Compute Canada job. §6.4 runs a 150-DA sliver to prove the chain connects end to end in-window.

## 6.1 Cross-basis builder

Quadratic B-spline on temperature, natural cubic spline on lag. Knots specified verbatim per Gasparrini 2022 supplement.

In [ ]:
build_crossbasis <- function(T_series, lag_max = 21) {
  T_knots <- quantile(T_series, probs = c(0.10, 0.75, 0.90), na.rm = TRUE)
  cb <- crossbasis(
    x      = T_series,
    lag    = lag_max,
    argvar = list(fun = "bs", degree = 2, knots = T_knots),
    arglag = list(fun = "ns", knots = logknots(lag_max, nk = 3))
  )
  stopifnot(attr(cb, "argvar")$fun    == "bs")
  stopifnot(attr(cb, "argvar")$degree == 2)
  cb
}

## 6.2 Strata factories + qAIC

Two stratification variants. Coarse (Variant A): DA × year × month, DOW as covariate. Fine (Variant B): DOW absorbed into the stratum. qAIC computed from deviance, not logLik — `gnm` with `eliminate` profiles out the stratum intercepts and does not expose a logLik, so the textbook −2·logLik form returns NA. Deviance form: deviance/φ̂ + 2k.

In [ ]:
make_strata_A <- function(DA_id, date) {
  factor(paste(DA_id, year(date), month(date), sep = "_"))
}

make_strata_B <- function(DA_id, date) {
  factor(paste(DA_id, year(date), month(date), wday(date), sep = "_"))
}

qaic <- function(fit) {
  phi <- summary(fit)$dispersion
  deviance(fit) / phi + 2 * length(coef(fit))
}

## 6.3 Stage 1 fit wrapper

For one (CMA, age) pair: fits both strata variants, returns the qAIC winner with coefs, vcov, strata counts, mean deaths per stratum.

Separates gnm's three outcomes: raised, ran without converging, converged. Only a converged fit reaches the qAIC pick. If neither variant converges, the function returns diagnostics rather than a winner.

`cb <- cb_template` binds the cross-basis into this function's frame, which is the environment the formula is built in. gnm resolves `cb` from there.

In [ ]:
fit_stage1 <- function(cma_age_data, cb_template, cma_label, age_label, verbose = TRUE) {
  if (verbose) cat(sprintf("\n=== Stage 1: %s, age %s ===\n", cma_label, age_label))

  cb <- cb_template

  cma_age_data[, strata_A := make_strata_A(DA_id, date)]
  cma_age_data[, strata_B := make_strata_B(DA_id, date)]
  cma_age_data[, dow := factor(wday(date))]
  cma_age_data[, t   := as.integer(date - min(date)) + 1]

  results <- list()
  for (variant in c("A", "B")) {
    strata_col <- if (variant == "A") "strata_A" else "strata_B"
    formula <- if (variant == "A") {
      n_deaths ~ cb + dow + ns(t, df = 20)
    } else {
      n_deaths ~ cb + ns(t, df = 20)
    }
    cma_age_data[, strata_use := get(strata_col)]

    fit <- try(gnm(formula, data = cma_age_data,
                   family = quasipoisson(), eliminate = strata_use),
               silent = TRUE)

    if (inherits(fit, "try-error")) {
      err <- conditionMessage(attr(fit, "condition"))
      if (verbose) cat(sprintf("  Variant %s: gnm RAISED -> %s\n", variant, err))
      results[[variant]] <- list(status = "raised", err = err)
      next
    }
    if (!isTRUE(fit$converged)) {
      if (verbose) cat(sprintf("  Variant %s: gnm RAN, converged FALSE (iterMax)\n", variant))
      results[[variant]] <- list(status = "noconv", fit = fit)
      next
    }

    results[[variant]] <- list(
      status   = "ok",
      fit      = fit,
      qaic     = qaic(fit),
      n_strata = length(unique(cma_age_data[[strata_col]])),
      mean_deaths_per_stratum = sum(cma_age_data$n_deaths) /
                                length(unique(cma_age_data[[strata_col]]))
    )
    if (verbose) cat(sprintf("  Variant %s: converged, qAIC = %.1f, strata = %d\n",
                             variant, results[[variant]]$qaic, results[[variant]]$n_strata))
  }

  ok <- names(results)[sapply(results, function(r) r$status == "ok")]
  if (!length(ok)) {
    if (verbose) cat("  BOTH VARIANTS FAILED — returning diagnostics, no fit\n")
    return(invisible(list(cma = cma_label, age = age_label, status = "failed", diag = results)))
  }

  qa <- sapply(ok, function(v) results[[v]]$qaic)
  winner_var <- ok[which.min(qa)]
  wfit <- results[[winner_var]]$fit
  cb_idx <- grep("^cb", names(coef(wfit)))
  stopifnot(length(cb_idx) == 25)

  list(
    cma             = cma_label,
    age             = age_label,
    status          = "ok",
    winner_variant  = winner_var,
    coef            = coef(wfit)[cb_idx],
    vcov            = vcov(wfit)[cb_idx, cb_idx],
    n_strata_A      = if (!is.null(results$A$n_strata)) results$A$n_strata else NA,
    n_strata_B      = if (!is.null(results$B$n_strata)) results$B$n_strata else NA,
    qaic_A          = if (!is.null(results$A$qaic)) results$A$qaic else Inf,
    qaic_B          = if (!is.null(results$B$qaic)) results$B$qaic else Inf,
    mean_dps_winner = results[[winner_var]]$mean_deaths_per_stratum,
    cb_template     = cb_template
  )
}

## 6.3b Assemble the model data

`gnm` resolves `cb` from the calling environment when it is not a column, so the cross-basis stays out of the frame. The model data carries vector columns only — `n_deaths`, `DA_id`, `date` — and `cb` sits alongside as a free-standing 25-column matrix.

Assigning a 25-column matrix into a `data.table` raises: `$<-` dispatches into `set()`, which refuses the shape. `:=` flattens it to one vector and the formula sees one term where it needs 25.

Rows pair by position and nothing else checks it—build both off the same object in the same order and assert `nrow(dt) == nrow(cb)` before the fit.

In [ ]:
dt <- data.table(n_deaths = sl$n_deaths, DA_id = sl$DA_id, date = sl$date)
cat("dt rows:", nrow(dt), " cb rows:", nrow(cb), " match:", nrow(dt) == nrow(cb), "\n")
cat("dt has cb col:", "cb" %in% names(dt), " (must be FALSE)\n")
cat("cb in global env:", exists("cb") && is.matrix(cb) && ncol(cb) == 25, "\n")

res_tor <- fit_stage1(dt, cb, "Toronto", "age_75_84")

cat("\nstatus:", res_tor$status, " winner:", res_tor$winner_variant, "\n")
cat("qAIC A:", round(res_tor$qaic_A,1), " B:", round(res_tor$qaic_B,1), "\n")
cat("strata A:", res_tor$n_strata_A, " B:", res_tor$n_strata_B, "\n")
cat("coef:", length(res_tor$coef), " vcov:", paste(dim(res_tor$vcov), collapse=" x "),
    " NA:", sum(is.na(res_tor$coef)), "\n")
cat("mean deaths/stratum (winner):", round(res_tor$mean_dps_winner,3), "\n")

Result (Toronto sliver, v2 DGP, 150 DAs, age 75–84, seed 42): both variants converged. Variant A qAIC 27,047.0 on 3,750 strata; Variant B qAIC 55,301.9 on 26,250 strata. Winner A, coef 25, vcov 25×25, no NA, mean deaths/stratum 0.645, dispersion 0.298. Deviance 9,136.5 → 8,042.0, converged at iteration 7.

## 6.4 Stage 1 on a Toronto sliver

150 random DAs, one age band (~115k rows) — small enough to fit in-window, enough to prove coef/vcov come out the shape Stage 2 needs. One function does the assembly for any city: pull the sliver index at seed 42, subset, attach each DA's temperature in matching row order, build the cross-basis, fit both variants, qAIC-pick. temp_mat rows are in da_idx order (imposed at build), so positional indexing is safe.

In [ ]:
fit_city_sliver <- function(sim_dt, temp_mat, cma_label, age = "age_75_84", n = 150, seed = 42) {
  set.seed(seed)
  idx <- sample(unique(sim_dt$da_idx), n)
  sl  <- sim_dt[da_idx %in% idx & age_band == age]
  sl[, DA_id := da_idx]
  setorder(sl, da_idx, date)
  tl <- data.table(
    da_idx = rep(idx, each = length(study_dates)),
    date   = rep(study_dates, times = n),
    temp_C = as.vector(t(temp_mat[idx, ]))
  )
  sl <- tl[sl, on = c("da_idx","date")]
  cb <- build_crossbasis(sl$temp_C, lag_max = 21)
  res <- fit_stage1(sl, cb, cma_label, age)
  list(sliver = sl, cb = cb, res = res)
}

tor_fit <- fit_city_sliver(sim,     temp_mat,     "Toronto",   "age_75_84")
mtl_fit <- fit_city_sliver(mtl_sim, mtl$temp_mat, "Montreal",  "age_75_84")
van_fit <- fit_city_sliver(van_sim, van$temp_mat, "Vancouver", "age_75_84")

for (f in list(tor_fit, mtl_fit, van_fit)) {
  r <- f$res
  cat(sprintf("%-10s winner %s  qAIC A %.1f  B %.1f  coef %d  NA %s\n",
              r$cma, r$winner_variant, r$qaic_A, r$qaic_B, length(r$coef), any(is.na(r$coef))))
}

"""Result: all three cross-bases 114,750 × 25 (5 temp × 5 lag), Variant A wins all three, no NA. qAIC A/B: Toronto 28,979 / 72,114, Montréal 29,000 / 72,135, Vancouver 27,253 / 75,012. A wins because the sparse sliver starves Variant B's DA×year×month×DOW strata (3,750 vs 26,250 strata) — qAIC penalises the near-empty fine strata. Vancouver's spread is distinct from the two continental cities (A lower, B higher): maritime climate, thinner heat signal, A fits the sparser response cheaper. A winning is sliver-specific — at full scale with real dense counts B may win, which is why both variants always run. Toronto dispersion 0.21 (underdispersed clean Poisson; real CVSD will run >1).

## 6.5 Simulate mortality — Vancouver

Fire the DGP on the VAN substrate. Same `simulate_counts` as §5.5/§9.6, pointed at `van`. Maritime climate → lower MMT than the continental cities. Run before §6.4's van_fit call (the sliver consumes van_sim).

In [ ]:
mmt_van <- apply(van$temp_mat, 1, function(x) quantile(x, 0.80, na.rm = TRUE))

van_da_long <- CJ(da_idx = 1:van$n_da, date = study_dates, age_band = names(annual_rates))
pop_long <- melt(van$da_age[, .(da_idx, age_0_64, age_65_74, age_75_84, age_85p)],
                 id.vars = "da_idx", variable.name = "age_band", value.name = "pop")
pop_long[, age_band := as.character(age_band)]
van_da_long <- pop_long[van_da_long, on = c("da_idx", "age_band")]
van_da_long[, annual_rate := annual_rates[age_band]]
van_da_long[, lambda0 := pop * annual_rate / 1000 / 365]

van_sim <- simulate_counts(van$temp_mat, van_da_long, van$truth_factors, mmt_van, seed = 42)

cat("van_sim rows:", nrow(van_sim), " NA pop:", sum(is.na(van_da_long$pop)),
    " total deaths:", sum(van_sim$n_deaths), "\n")
cat("mmt_van range:", paste(round(range(mmt_van),1), collapse="-"), "\n")
print(van_sim[, .(deaths = sum(n_deaths)), by = age_band])

Result: 10,933,380 rows (3573 x 765 x 4), zero NA pop. 55,265 deaths over 5 warm-seasons. Deaths climb by age band: 2,748 -> 13,412 -> 16,951 -> 22,154 (0-64 -> 85+). mmt_van 17.4-20.7°C — below the continental cities (maritime). Deaths track population, not DA count: VAN's 3,573 DAs carry fewer deaths per DA (15.5) than MTL's 6,504 (22.0), so the total sits well below a DA-count scale of MTL.

# 7. Reduce + Stage 2 Meta-regression

Reduce the 25-dim cross-basis to its 5-dim overall cumulative curve (Gasparrini-Armstrong 2013) before pooling — the parameter-count argument forces this: a full 25-dim Ψ has 25·26/2 = 325 unique elements; the reduced 5-dim Ψ has 15. Pooling the full cross-basis is infeasible; the reduced curve is what Stage 2 meta-regresses.

## 7.1 Reduce function

`crossreduce(type = "overall")` collapses the 25-dim cross-basis to its 5-dim curve. `model.link = "log"` required — with raw coef/vcov there's no model object to read the link from. Recenter at CMA-median; DA-specific re-centering in Stage 3.

In [ ]:
reduce_fit <- function(stage1_result, ref_temp, verbose = TRUE) {
  cb_t <- stage1_result$cb_template
  red  <- crossreduce(
    basis = cb_t,
    coef  = stage1_result$coef,
    vcov  = stage1_result$vcov,
    model.link = "log",
    type  = "overall",
    cen   = ref_temp
  )

  list(
    cma         = stage1_result$cma,
    age         = stage1_result$age,
    theta_star  = coef(red),
    V_star      = vcov(red),
    cen         = ref_temp,
    reduced_obj = red
  )
}

## 7.2 mixmeta wrapper

Formula and random structure scale with N. PCs need >3 between-CMA d.f. (3 CMAs give 2 after the intercept — collinear, mixmeta errors), so they drop below the national scale and return only when 164/15 clears the ≥10 threshold. age_band needs >1 band. The random ~1|CMA with a 5-dim Ψ is unestimable from few groups; REML is tried, fixed-effects is the fallback. At pilot N_CMA=3, one band: intercept-only, REML if Ψ holds.

In [ ]:
fit_stage2 <- function(reduced_list, cma_predictors_df = NULL,
                       use_pcs = NULL, use_age = NULL, verbose = TRUE) {
  n_obs     <- length(reduced_list)
  theta_mat <- t(sapply(reduced_list, function(x) x$theta_star))
  colnames(theta_mat) <- paste0("theta", seq_len(ncol(theta_mat)))
  V_list    <- lapply(reduced_list, function(x) x$V_star)

  pred_df <- data.table(
    CMA      = sapply(reduced_list, function(x) x$cma),
    age_band = sapply(reduced_list, function(x) x$age),
    obs_idx  = seq_len(n_obs)
  )
  if (!is.null(cma_predictors_df)) pred_df <- merge(pred_df, cma_predictors_df, by = "CMA", all.x = TRUE)
  setorder(pred_df, obs_idx)
  pred_df <- cbind(pred_df, as.data.table(theta_mat))

  n_cma  <- length(unique(pred_df$CMA))
  n_band <- length(unique(pred_df$age_band))
  if (is.null(use_pcs)) use_pcs <- (n_cma > 4 && all(c("PC1","PC2","PC3") %in% names(pred_df)))
  if (is.null(use_age)) use_age <- (n_band > 1)

  rhs <- c(if (use_age) "age_band", if (use_pcs) c("PC1","PC2","PC3"))
  rhs <- if (length(rhs)) paste(rhs, collapse = " + ") else "1"
  lhs <- sprintf("cbind(%s)", paste(colnames(theta_mat), collapse = ", "))
  form <- as.formula(sprintf("%s ~ %s", lhs, rhs))
  use_random <- n_cma >= 3

  fit <- tryCatch(
    mixmeta(form, S = V_list, data = pred_df, method = "reml",
            random = if (use_random) ~ 1 | CMA else NULL),
    error = function(e) { if (verbose) cat("  REML failed:", conditionMessage(e), "-> fixed\n"); NULL }
  )
  method_used <- "reml"
  if (is.null(fit)) {
    fit <- mixmeta(form, S = V_list, data = pred_df, method = "fixed")
    method_used <- "fixed"
  }

  if (verbose) cat(sprintf("  formula: %s | random: %s | method: %s\n",
                           deparse(form), use_random && method_used == "reml", method_used))
  list(fit = fit, pred_df = pred_df, theta_mat = theta_mat, V_list = V_list,
       method = method_used, formula = form)
}

## 7.3 Reduce the sliver fit (25 → 5)

Collapse the lag dimension of the Stage 1 sliver fit into its overall cumulative curve — the object Stage 2 pools. Centered at the sliver's median temperature.

In [ ]:
ref_temp <- median(sliver$temp_C)
red <- reduce_fit(res, ref_temp)

cat("ref_temp:", round(ref_temp,1), "\n")
cat("theta_star:", length(red$theta_star),
    " V_star:", paste(dim(red$V_star), collapse=" x "),
    " any NA:", any(is.na(red$theta_star)), "\n")

"""Result: ref_temp 19.4°C. theta_star is a 5-vector, V_star 5×5, no NA — the 25-dim Stage 1 output collapsed to the 5-dim reduced curve. The same reduce runs for all three cities, each centered at its own median: Toronto 19.4, Montréal 19.0, Vancouver 17.0°C. The 2.4°C gap between Toronto and maritime Vancouver pre-loads the Criterion 2 cross-CMA contrast. These three 5-vectors are what Stage 2 pools.

7.4 Stage 2 — the first non-degenerate pool

Reduce all three slivers at each city's own median, stack into a reduced_list, pool with fit_stage2. Three CMAs, one age band: the wrapper drops PCs (no between-CMA d.f.) and age_band (one band), leaving an intercept-only multivariate meta with a random intercept per CMA. REML estimates the between-city Ψ; Cholesky confirms positive-definite. This retires the N_CMA=1 fallback — the first real pool.

In [ ]:
reduced_tor <- reduce_fit(tor_fit$res, median(tor_fit$sliver$temp_C))
reduced_mtl <- reduce_fit(mtl_fit$res, median(mtl_fit$sliver$temp_C))
reduced_van <- reduce_fit(van_fit$res, median(van_fit$sliver$temp_C))

reduced_list <- list(reduced_tor, reduced_mtl, reduced_van)
s2 <- fit_stage2(reduced_list)

stage2     <- s2$fit
reduced_df <- s2$pred_df
vcov_list  <- s2$V_list
chol_ok    <- tryCatch({ chol(stage2$Psi); TRUE }, error = function(e) FALSE)

cat("method:", s2$method, " converged:", isTRUE(stage2$converged),
    " pooled coef:", length(coef(stage2)), " Cholesky:", chol_ok, "\n")
print(round(coef(stage2), 3))

Result: REML converged, Ψ positive-definite (Cholesky passes). Pooled 5-vector [0.945, -1.986, -0.081, -4.688, 7.110] — the across-city average reduced curve, inverse-vcov weighted across Toronto, Montréal, Vancouver. Three groups carried a 5-dim Ψ where the parameter count (3/15) predicted they could not — REML held on the better of the two paths, no fixed-effects fallback needed. First non-degenerate Stage 2 in the project; the N_CMA=1 fallback retires. This is the object Stage 3 downscales from.

## 7.5 Stage 2 at K=5 — PC predictors enter

CMA-level predictors are the column means of each city's Z — one 17-vector per city, 5 × 17 — then `prcomp(scale. = TRUE)` with PC1–3 as meta-regressors. Five cities clears `fit_stage2`'s `n_cma > 4` gate, so the formula is `cbind(θ1..θ5) ~ PC1 + PC2 + PC3`.

At 5 observations on 17 variables the PCA carries at most 4 non-zero components, so cumulative variance is not informative at this K. Per-PC split 0.431 / 0.336 / 0.233.

In [ ]:
Z_list <- list(Toronto = tor_out$Z, Montreal = mtl_out$Z, Vancouver = van_out$Z,
               Ottawa = ott_out$Z, Quebec = qc_out$Z)

Z_means <- t(sapply(Z_list, colMeans))
stopifnot(nrow(unique(Z_means)) == nrow(Z_means))

pca_cma <- prcomp(Z_means, scale. = TRUE)
cma_predictors <- data.table(CMA = rownames(Z_means),
                             PC1 = pca_cma$x[,1], PC2 = pca_cma$x[,2], PC3 = pca_cma$x[,3])

reduced_list_5 <- list(tor_out$red, mtl_out$red, van_out$red, ott_out$red, qc_out$red)
s2_k5     <- fit_stage2(reduced_list_5, cma_predictors_df = cma_predictors)
stage2_k5 <- s2_k5$fit
chol_ok   <- tryCatch({ chol(stage2_k5$Psi); TRUE }, error = function(e) FALSE)

cat("method:", s2_k5$method, " converged:", isTRUE(stage2_k5$converged), "\n")
cat("cholesky:", chol_ok, " df.residual:", stage2_k5$df.residual, "\n")
cat("n coef:", length(coef(stage2_k5)), " (expect 20 = 4 terms x 5 outcomes)\n")
print(round(coef(stage2_k5), 3))
print(cma_predictors)

# 8. Stage 3 Downscale + Figure 3

## 8.1 DA-level PCA

`prcomp(scale. = TRUE)` on the 7,694 × 17 matrix. First 3 PCs should explain >50% (assertion enforced).

In [ ]:
fit_da_pca <- function(Z_matrix, da_ids, verbose = TRUE) {
  pca <- prcomp(Z_matrix, scale. = TRUE)
  var_explained <- summary(pca)$importance["Proportion of Variance", 1:3]
  cum_var       <- summary(pca)$importance["Cumulative Proportion", 3]

  da_scores <- data.table(
    DAUID = da_ids,
    PC1   = pca$x[, 1],
    PC2   = pca$x[, 2],
    PC3   = pca$x[, 3]
  )
  if (verbose) cat(sprintf("PCA: cum var first 3 = %.1f%%\n", 100*cum_var))
  stopifnot(cum_var > 0.5)
  list(pca = pca, scores = da_scores, var_explained = var_explained)
}

## 8.2 DA-level theta prediction

Per-DA θ from the K=5 pool. The 20 coefficients reshape to 5×4 — intercept + PC1–3 per curve term — and each DA's projected PC scores multiply through, returning a distinct 5-vector per DA. Scores must come from the pca_cma projection (§10.1), not fit_da_pca's own rotation; the two are permuted and mixed. Superseded 2026-08-12: §11's bottom-up form feeds fit_da_pca scores natively; the projection rule applied to the top-down spine only. Replaced 2026-08-05: the intercept-only stamp raised on the 20-vector and was the birthplace of the flat map.

In [ ]:
predict_da_theta <- function(stage2_obj, pc_scores, da_ids, band = "age_75_84") {
  cf <- coef(stage2_obj)
  stopifnot(length(cf) == 20, nrow(pc_scores) == length(da_ids))
  B  <- matrix(cf, nrow = 5)
  th <- cbind(1, pc_scores) %*% t(B)
  colnames(th) <- paste0("theta", 1:5)
  data.table(DAUID = da_ids, age_band = band, th)
}

## 8.3 Two-pass MMT (U11)

Two-pass MMT. First pass: predict at the CMA median, take the DA-specific MMT. Second pass: re-centre at that MMT for final RR/AF — MMT is local to each area (Gasparrini et al. 2022). The reduced curve is 5-dim, so predict the 5-vector through a `onebasis` rebuilt from `attr(cb_template, "argvar")`, with `vcov = diag(1e-8, 5)`.

In [ ]:
compute_da_mmt <- function(da_theta_row, cb_template, temp_range_da, cma_median) {
  av <- attr(cb_template, "argvar")
  red_basis <- onebasis(temp_range_da, fun = av$fun, degree = av$degree, knots = av$knots)
  pred1 <- crosspred(basis = red_basis,
                     coef  = da_theta_row,
                     vcov  = diag(1e-8, 5),
                     at    = temp_range_da,
                     cen   = cma_median)
  pred1$predvar[which.min(pred1$allfit)]
}

## 8.4 Monte Carlo CI at fixed-effect level (U12, Masselot 2025)

Sample 1000 draws of Stage 2 coefs from N(β̂, V̂), propagate per-DA, take quantiles aggregate. Naive per-DA MC underestimates aggregate CIs because it ignores covariance across DAs induced by shared Stage 2 fixed effects.

In [ ]:
monte_carlo_ci <- function(stage2_obj, da_scores, n_sim = 1000, verbose = TRUE) {
  beta_hat <- as.vector(coef(stage2_obj$fit))
  V_hat    <- vcov(stage2_obj$fit)
  beta_sims <- MASS::mvrnorm(n_sim, mu = beta_hat, Sigma = V_hat)
  if (verbose) cat(sprintf("MC: %d × %d draws\n", nrow(beta_sims), ncol(beta_sims)))
  beta_sims
}

## 8.5 Direct standardization (Canadian 2011 std pop, U14)

StatCan/CIHI standard. ESP 2013 is the sensitivity in the appendix.

In [ ]:
cdn_2011_std <- c(
  age_0_64  = 0.8210,
  age_65_74 = 0.0930,
  age_75_84 = 0.0580,
  age_85p   = 0.0280
)
stopifnot(abs(sum(cdn_2011_std) - 1) < 1e-6)

standardize_da_rate <- function(da_age_rates) {
  rates_wide <- dcast(da_age_rates, DAUID ~ age_band, value.var = "rate")
  std_rate   <- with(rates_wide,
                     cdn_2011_std["age_0_64"]  * age_0_64  +
                     cdn_2011_std["age_65_74"] * age_65_74 +
                     cdn_2011_std["age_75_84"] * age_75_84 +
                     cdn_2011_std["age_85p"]   * age_85p)
  data.table(DAUID = rates_wide$DAUID, std_rate = std_rate)
}

## 8.6 Choropleth (no significance overlay, U13)

Continuous color scale, no stippling. Per-DA significance overlays at ~7,000+ DA scale generate false-positive theater from multiple comparisons.

In [ ]:
make_choropleth <- function(toronto_sf, value_col, title_str, palette = "magma") {
  ggplot(toronto_sf) +
    geom_sf(aes_string(fill = value_col), color = NA) +
    scale_fill_viridis_c(option = palette, na.value = "grey80") +
    theme_minimal(base_size = 11) +
    labs(title = title_str, fill = NULL) +
    theme(panel.grid = element_blank(),
          axis.text  = element_blank(),
          axis.title = element_blank())
}

## 8.7 Criterion 3 - PCA recovery

Empricial PCA loadings vs the simulation's true loading matrix L. Pass if each true factor maps to a distinct PC at |cor| > 0.7. Greedy 1-to-1 assignment by |cor| across the 6 permutations.

In [ ]:
pca_out  <- fit_da_pca(Z, truth_factors$DAUID)
emp_load <- pca_out$pca$rotation[, 1:3]
cormat   <- cor(emp_load, true_loadings)

perms    <- rbind(c(1,2,3), c(1,3,2), c(2,1,3), c(2,3,1), c(3,1,2), c(3,2,1))
best_p   <- perms[which.max(apply(perms, 1, function(p) mean(abs(diag(cormat[p, ]))))), ]
diag_cor <- abs(diag(cormat[best_p, ]))

cat("matched |cor| per factor:", round(diag_cor, 3), "\n")
cat("mean |cor|:", round(mean(diag_cor), 3), "  threshold 0.70\n")
cat("per-true-F max |cor|:", round(apply(abs(cormat), 2, max), 3), "\n")

Result: Mean |cor| 0.731, above the 0.70 threshold. F1 and F2 map to distinct PCs at 0.85 each. F3 maps to 0.49 — its strongest correlation (0.75) is with the PC that F1 already holds. F3 and F1 do not separate. DGP fix in open threads.

## 8.8 The flat "before" panel — recovered vs DGP truth

Left: DGP true heat-vulnerability `1 + 0.4·F1 + 0.2·F3` per DA, from `truth_factors`—real spread (SD 0.447, range −0.58 to 2.78). Right: the recovered map—stamp the intercept-only pool through the reduced temperature basis, read heat RR at the 99th percentile, one value on every DA. Flat by construction. The recovered RR level is a placeholder magnitude

Predict the 5-vector through the reduced basis (`onebasis` from `attr(cb_template, "argvar")`). Truth here is the additive `1 + 0.4·F1 + 0.2·F3`; under the exp DGP (§5.3) the consistent target is `exp(0.4·F1 + 0.2·F3)`.

Status 2026-07-29: recomputed under v2 against the K=5 pool. The v1 result below is retained for provenance; the v2 baseline follows it.

Map: The truth colours are iid `rnorm` draws—spread *without* spatial structure, i.e. not a vulnerability geography but *between-CMA* structure, not between-DA spatial pattern; within a city the factors stay iid.

Baseline 2026-07-29 (v2, K=5 pool, Toronto substrate seed 77). `predict_da_theta` verbatim raises: `length(cf) == 5 is not TRUE` against a 20-vector. Sliced to the five `(Intercept)` terms by name — [−0.872, −4.606, −4.481, −3.813, −1.020] — and predicted through the 5-column basis with the link declared: heat_p 27.19°C, rr_flat 3.176, MMT 21.68°C (inside range), RR 81.7 at the cold end where `bs` drops its first column and `allfit(min) = 0` by construction.

Recovered: 7,682 rows, 1 distinct value, SD exactly 0 (`identical(sd(x), 0)` TRUE). Not small — hard zero. One scalar recycled. The flatness is born in §8.2 where the pool is stamped, not downstream in the basis; a small-but-nonzero SD would have placed the fault in the basis instead. Truth SD 0.4454 (√0.20 = 0.4472, gap 0.5 SE at n = 7,682), mean 0.946 against v1's 1.00 — μ enters the mean, not the SD. Target `exp(0.4·F1 + 0.2·F3)` SD 0.5002.

Status 2026-08-05: superseded. §10 rules the axis and replaces the stamp; recovered SD is non-zero. Retained as the flat baseline.

θ₅ carries V_star diagonal 37.4, SE 6.1, against 1.4–2.0 for θ₁–θ₄. The heat end is the thinnest part of the curve and it is where amplitude is read. RR runs 3.18 at the 99th percentile to 29.46 at the maximum — a factor of 9 in the last 3°C. Read at the percentile, not the max.

Toronto temperature: full city 2.2–30.15°C across 7,682 DAs. The 3.68–30.09 recorded elsewhere is the 150-DA sliver. The hot end is unchanged; the cold tail is a coverage artifact of 2% sampling, not a different series.

In [ ]:
truth_vuln <- data.table(
  DAUID = truth_factors$DAUID,
  truth = 1 + 0.4 * truth_factors$F1 + 0.2 * truth_factors$F3
)

cen_tor <- red5$Toronto$cen
av <- attr(res5$Toronto$cb_template, "argvar")
red_basis <- onebasis(temp_grid <- seq(min(temp_mat), max(temp_mat), length.out = 100),
                      fun = av$fun, degree = av$degree, knots = av$knots)  # 5-dim reduced basis
cf <- as.numeric(coef(stage2_k5)[grep("\\(Intercept\\)$", names(coef(stage2_k5)))])
stopifnot(length(cf) == 5)
pred_flat <- crosspred(basis = red_basis, coef = cf, vcov = red5$Toronto$V_star,
                       model.link = "log", at = temp_grid, cen = cen_tor)
heat_p  <- quantile(temp_mat, 0.99)
rr_flat <- approx(pred_flat$predvar, pred_flat$allRRfit, xout = heat_p)$y
recovered <- data.table(DAUID = truth_factors$DAUID, recovered_rr = rr_flat)

cat("truth aligned:", all(truth_vuln$DAUID == truth_factors$DAUID), "\n")
cat("truth: sd", round(sd(truth_vuln$truth),3),
    " range", paste(round(range(truth_vuln$truth),2), collapse=" "), "\n")
cat("recovered rr: sd", round(sd(recovered$recovered_rr),6), " val", round(rr_flat,4), "\n")

In [ ]:
tor_sf <- st_read(file.path(DRIVE, "toronto_da.geojson"), quiet = TRUE)
tor_sf$DAUID <- as.character(tor_sf$DAUID)
tor_sf <- merge(tor_sf, truth_vuln, by = "DAUID", all.x = TRUE)   # key-type match or all rows go grey
tor_sf <- merge(tor_sf, recovered,  by = "DAUID", all.x = TRUE)

cat("polys:", nrow(tor_sf), " truth attached:", sum(!is.na(tor_sf$truth)),
    " recovered attached:", sum(!is.na(tor_sf$recovered_rr)), "\n")

p_truth <- ggplot(tor_sf) +
  geom_sf(aes(fill = truth), color = NA) +
  scale_fill_viridis_c(option = "magma", na.value = "grey85") +
  theme_minimal(base_size = 10) +
  labs(title = "DGP truth", fill = NULL) +
  theme(panel.grid = element_blank(), axis.text = element_blank(), axis.title = element_blank())

p_recov <- ggplot(tor_sf) +
  geom_sf(aes(fill = recovered_rr), color = NA) +
  scale_fill_viridis_c(option = "magma", na.value = "grey85") +
  theme_minimal(base_size = 10) +
  labs(title = "Recovered", fill = NULL) +
  theme(panel.grid = element_blank(), axis.text = element_blank(), axis.title = element_blank())

ggsave("/content/before_truth.png",     p_truth, width = 6, height = 6, dpi = 130)
ggsave("/content/before_recovered.png", p_recov, width = 6, height = 6, dpi = 130)

Result: polys 7716, truth + recovered each attached to 7682 (geojson untrimmed; 34 water/zero-pop DAs map grey). Truth SD 0.447, range −0.58 to 2.78. Recovered SD 0, flat value 9.60.

"""Result (v1, additive, N_CMA=3): polys 7716, truth + recovered each attached to 7682 (geojson untrimmed; 34 water/zero-pop DAs map grey). Truth SD 0.447, range −0.58 to 2.78. Recovered SD 0.

The "flat value 9.60" recorded here was not a risk ratio. `crosspred` on raw coef/vcov without `model.link` returns `allfit` and no `allRRfit`; `approx(predvar, NULL)` then reads `temp_grid` against its own index. 9.60 = 2.2 + 26.19 × 0.2823, the grid evaluated at index 27.19. Confirmed 2026-07-29: the same code under v2, whose θ vector shares no sign or magnitude with v1's, returns 9.5955. A risk ratio that does not move when every coefficient changes is not a risk ratio. Recovered SD 0 stands — only the magnitude was fiction.

# 9. Scaling - Montréal + Vancouver to bronze

Ports the full Toronto chain to two more CMAs to reach N_CMA = 3. Geography (§9.1), population (§9.2), temperature (§9.3), then one shared substrate builder (§9.5) that aligns latents, observed Z, and temperature per city. Output: two bronze-ready substrates. Stage 2 consumes three reduced curves; this section supplies the two new ones' inputs.

## 9.1 Geography — DGRF filter + shapefile join

Filter DGRF to each CMA DGUID, dedup to DA level, join to national `da_all`.

In [ ]:
cma_dguids <- c(Montreal = "2021S0503462", Vancouver = "2021S0503933")

cma_da_lists <- lapply(cma_dguids, function(dg) {
  db_rows <- dgrf[CMADGUID_RMRIDUGD == dg]
  dauids  <- substr(unique(db_rows$DADGUID_ADIDUGD), 10, 17)
  list(n_db = nrow(db_rows), dauids = dauids)
})

cma_polys <- lapply(names(cma_da_lists), function(nm) {
  dauids <- cma_da_lists[[nm]]$dauids
  poly   <- da_all[da_all$DAUID %in% dauids, ]
  list(poly = poly, n = nrow(poly),
       missing = length(setdiff(dauids, poly$DAUID)),
       area = round(sum(poly$LANDAREA), 1))
})
names(cma_polys) <- names(cma_da_lists)

for (nm in names(cma_polys)) {
  p <- cma_polys[[nm]]; d <- cma_da_lists[[nm]]
  cat(sprintf("%-10s DAs: %5d  missing: %d  area: %7.1f km2  prefix: %s\n",
              nm, p$n, p$missing, p$area,
              paste(unique(substr(d$dauids, 1, 2)), collapse=",")))
}

Result: Montréal 6,574 DAs, all prefix 24 (Quebec). Vancouver 3,590 DAs, all prefix 59 (BC). Zero missing on both. Land areas 4,670 / 2,879 km². Saved to saves_mtlvan_clean.tar.gz (4.68 MB).

## 9.2 Population — Census Profile per CMA

Download each province's Census Profile server-side with wget (browser upload truncates ~800 MB silently). Verify GEONO at source before trusting. Filter to each CMA's DAs with streaming bash grep, extract the 18 age IDs, aggregate to 4 bands. §2.2–2.4 logic per city.

In [ ]:
prof_dir <- "/content/statcan"
base_url <- "https://www12.statcan.gc.ca/census-recensement/2021/dp-pd/prof/details/download-telecharger/comp/GetFile.cfm?Lang=E&FILETYPE=CSV&GEONO="
geono    <- c(montreal = "006_Quebec", vancouver = "006_BC_CB")
prov_sfx <- c(montreal = "Quebec", vancouver = "BritishColumbia")

for (city in names(geono)) {
  dest <- sprintf("%s/profile_%s.zip", prof_dir, city)
  if (!file.exists(dest)) system(sprintf('wget -q -O "%s" "%s%s"', dest, base_url, geono[city]))
  ex_dir <- sprintf("%s/profile_%s_extracted", prof_dir, city)
  if (!dir.exists(ex_dir)) { dir.create(ex_dir); unzip(dest, exdir = ex_dir) }
}

age_ids  <- c(10,11,12,14,15,16,17,18,19,20,21,22,23, 25,26, 27,28, 29)
band_map <- data.table(
  CHARACTERISTIC_ID = age_ids,
  band = c(rep("age_0_64",13), rep("age_65_74",2), rep("age_75_84",2), rep("age_85p",1))
)

cma_age_data <- list()
for (city in names(geono)) {
  src <- sprintf("%s/profile_%s_extracted/98-401-X2021006_English_CSV_data_%s.csv",
                 prof_dir, city, prov_sfx[city])
  dauids <- cma_da_lists[[ tools::toTitleCase(city) ]]$dauids
  pat <- sprintf("%s/%s_patterns.txt", prof_dir, city)
  out <- sprintf("%s/profile_%s_filtered.csv", prof_dir, city)
  writeLines(paste0('"', dauids, '"'), pat)
  if (!file.exists(out)) {
    system(sprintf('head -1 "%s" > "%s"', src, out))
    system(sprintf('grep -F -f "%s" "%s" >> "%s"', pat, src, out))
  }

  prof <- fread(out, select = c("ALT_GEO_CODE","CHARACTERISTIC_ID","C1_COUNT_TOTAL"))
  prof <- merge(prof[CHARACTERISTIC_ID %in% age_ids], band_map, by = "CHARACTERISTIC_ID")
  prof[is.na(C1_COUNT_TOTAL), C1_COUNT_TOTAL := 0]
  da_wide <- dcast(prof[, .(pop = sum(C1_COUNT_TOTAL)), by = .(ALT_GEO_CODE, band)],
                   ALT_GEO_CODE ~ band, value.var = "pop")
  da_wide[, total := age_0_64 + age_65_74 + age_75_84 + age_85p]
  da_wide[, ALT_GEO_CODE := as.character(ALT_GEO_CODE)]
  cma_age_data[[city]] <- da_wide
  cat(sprintf("%-10s DAs: %d  pop: %s\n", city, nrow(da_wide),
              format(sum(da_wide$total), big.mark=",")))
}

Result: Montréal 6,574 DAs / 4,290,150. Vancouver 3,590 DAs / 2,641,710. Both match published CMA totals. GEONO verified at source: 006_Quebec→quebec, 006_BC_CB→bc_cb.

## 9.3 Export polygons + derive bounding boxes

Write each CMA's polygons as a flat shapefile bundle (`-j` strips paths so EE sees a clean bundle). Bounding box derived from the polygons with `st_bbox` in WGS84.

In [ ]:
for (nm in names(cma_polys)) {
  poly <- cma_polys[[nm]]$poly
  poly$DAUID <- as.character(poly$DAUID)
  out_dir <- sprintf("/content/%s_da_shp", tolower(nm))
  dir.create(out_dir, showWarnings = FALSE)
  st_write(poly["DAUID"], file.path(out_dir, sprintf("%s_da.shp", tolower(nm))),
           delete_dsn = TRUE, quiet = TRUE)
  zp <- sprintf("%s/%s_da_shp.zip", DRIVE, tolower(nm))
  if (file.exists(zp)) file.remove(zp)
  zip(zipfile = zp, files = list.files(out_dir, full.names = TRUE), flags = "-j")

  poly_wgs <- sf::st_transform(poly, 4326); bb <- sf::st_bbox(poly_wgs)
  cat(sprintf("%-10s zip entries: %d  bbox: [%.2f, %.2f, %.2f, %.2f]\n",
              nm, length(unzip(zp, list = TRUE)$Name),
              bb["xmin"]-0.05, bb["ymin"]-0.05, bb["xmax"]+0.05, bb["ymax"]+0.05))
}

Result: montreal_da (6,574) / vancouver_da (3,590), 4 entries each. Boxes: MTL [-74.38, 45.17, -73.08, 46.02], VAN [-123.48, 48.95, -122.36, 49.62]. Upload both to EE manually (Code Editor → Assets → New → Shapefile → asset id montreal_da / vancouver_da), then verify asset feature counts equal the DA counts before reducing.

## 9.4 EE reduce-by-region (Python)

Separate kernel — Daymet V4 filtered to warm-season 2015–2019, mean tmax/tmin per DA per day, exported to Drive. Server-side on Google's cluster, ~15 min each.

In [ ]:
import ee
ee.Initialize(project='gen-lang-client-0569405164')

city_boxes = {
    "montreal_da":  ee.Geometry.Rectangle([-74.38, 45.17, -73.08, 46.02]),
    "vancouver_da": ee.Geometry.Rectangle([-123.48, 48.95, -122.36, 49.62]),
}
prefix = {"montreal_da": "montreal", "vancouver_da": "vancouver"}

for city, bbox in city_boxes.items():
    da_fc = ee.FeatureCollection(f'projects/gen-lang-client-0569405164/assets/{city}')
    daymet = (ee.ImageCollection('NASA/ORNL/DAYMET_V4')
              .filterDate('2015-05-01', '2019-10-01')
              .filterBounds(bbox).select(['tmax', 'tmin']))
    def keep_warm(img):
        m = ee.Date(img.get('system:time_start')).get('month')
        return img.set('keep', m.gte(5).And(m.lte(9)))
    warm = daymet.map(keep_warm).filter(ee.Filter.eq('keep', 1))
    def reduce_one_day(img):
        d = ee.Date(img.get('system:time_start')).format('YYYY-MM-dd')
        return img.reduceRegions(collection=da_fc, reducer=ee.Reducer.mean(),
                                 scale=1000, tileScale=4).map(lambda f: f.set('date', d))
    out = warm.map(reduce_one_day).flatten()
    ee.batch.Export.table.toDrive(
        collection=out, description=f'{prefix[city]}_daymet_2015_2019',
        folder='thesis/dlnm-pilot', fileNamePrefix=f'{prefix[city]}_daymet_2015_2019',
        fileFormat='CSV', selectors=['DAUID','date','tmax','tmin']).start()

Result: MTL ~5.0M rows, VAN ~2.75M rows, CSVs to Drive. VAN zero NA (no water-only DAs). MTL 25,245 raw NA → 28 all-water DAs, 0 partial.

## 9.5 Substrate builder — latents, Z, temperature per city

One function per city: filter pop>0, pivot temperature to a DA×day matrix, align row order to the DA list with `match()`, sample per-city latents (seed = 42 + province prefix, reproducible but distinct), build observed Z from the shared loading matrix L. Drop all-NA water DAs in lockstep across temp_mat, truth_factors, Z, da_age — baked into the function so a clean city can't hide the missing step.

In [ ]:
build_city_sim_substrate <- function(temp_dt, da_age_city, L, seed) {
  set.seed(seed)
  da_age_c <- as.data.table(da_age_city)[total > 0]
  temp_wide <- dcast(temp_dt, DAUID ~ date, value.var = "tmean_C")
  common <- intersect(da_age_c$ALT_GEO_CODE, temp_wide$DAUID)
  da_age_c  <- da_age_c[ALT_GEO_CODE %in% common]
  temp_wide <- temp_wide[DAUID %in% common][match(da_age_c$ALT_GEO_CODE, DAUID), ]
  temp_mat  <- as.matrix(temp_wide[, -1]); rownames(temp_mat) <- temp_wide$DAUID

  dead <- which(rowSums(is.na(temp_mat)) == ncol(temp_mat))
  if (length(dead)) {
    keep <- setdiff(seq_len(nrow(temp_mat)), dead)
    temp_mat <- temp_mat[keep, ]; da_age_c <- da_age_c[keep, ]
  }

  n  <- nrow(da_age_c)
  tf <- data.table(DAUID = da_age_c$ALT_GEO_CODE, da_idx = 1:n,
                   F1 = rnorm(n), F2 = rnorm(n), F3 = rnorm(n))
  Zc <- as.matrix(tf[, .(F1,F2,F3)]) %*% t(L) + matrix(rnorm(n*17, sd = 0.3), nrow = n)
  colnames(Zc) <- paste0("vuln", sprintf("%02d", 1:17))
  da_age_c[, da_idx := 1:n]
  list(temp_mat = temp_mat, truth_factors = tf, Z = Zc, da_age = da_age_c, n_da = n)
}

read_daymet <- function(csv) {
  d <- fread(csv, select = c("DAUID","date","tmax","tmin"))
  d[, `:=`(DAUID = as.character(DAUID), date = as.Date(date), tmean_C = (tmax+tmin)/2)]
  d[]
}

van <- build_city_sim_substrate(read_daymet(sprintf("%s/vancouver_daymet_2015_2019.csv", DRIVE)),
                                cma_age_data$vancouver, L, seed = 42 + 59)
mtl <- build_city_sim_substrate(read_daymet(sprintf("%s/montreal_daymet_2015_2019.csv", DRIVE)),
                                cma_age_data$montreal, L, seed = 42 + 24)

for (nm in c("van","mtl")) {
  s <- get(nm)
  cat(sprintf("%s  n_da: %d  temp_mat: %s  align: %s  NA: %d\n",
              nm, s$n_da, paste(dim(s$temp_mat), collapse="x"),
              all(rownames(s$temp_mat) == s$truth_factors$DAUID), sum(is.na(s$temp_mat))))
}

Result: VAN 3,573 DAs (17 zero-pop dropped, 0 water). MTL 6,504 DAs (zero-pop + 28 water dropped). Both aligned, zero NA. Two bronze-ready substrates. Bronze each (§6.4 logic, 150-DA sliver, age 75–84) → reduce each (§7.3) → three reduced curves with Toronto → Stage 2.

"""## 9.5b DGP patch — per-CMA per-factor latent offset (v2 builder)

`build_city_sim_substrate` draws `F1/F2/F3 ~ rnorm(n)` — iid, mean-zero, identical every city. Between-CMA variance ≈ 0, so the Stage 2 vulnerability slope is unidentified and the downscaled map is flat at any N. Per city, draw a per-factor offset `μ ~ N(0, 0.6)` once, then `F ~ N(μ, 1)` per DA. That injects the between-CMA variation Stage 2 reads.

Demo: two city seeds give two distinct μ triples, and F-means land on μ not 0 — seed 66 μ [1.394, 0.130, 0.251] → F-means [1.392, 0.092, 0.267]; seed 101 μ [-0.196, 0.331, -0.405] → [-0.182, 0.328, -0.392].

Seed key is the CMA code, not the province prefix. Stack all cities' μ and check `sum(duplicated(...)) == 0` before pooling.

In [ ]:
build_city_sim_substrate_v2 <- function(temp_dt, da_age_city, L, seed, mu_sd = 0.6) {
  set.seed(seed)
  da_age_c <- as.data.table(da_age_city)[total > 0]
  temp_wide <- dcast(temp_dt, DAUID ~ date, value.var = "tmean_C")
  common <- intersect(da_age_c$ALT_GEO_CODE, temp_wide$DAUID)
  da_age_c  <- da_age_c[ALT_GEO_CODE %in% common]
  temp_wide <- temp_wide[DAUID %in% common][match(da_age_c$ALT_GEO_CODE, DAUID), ]
  temp_mat  <- as.matrix(temp_wide[, -1]); rownames(temp_mat) <- temp_wide$DAUID

  dead <- which(rowSums(is.na(temp_mat)) == ncol(temp_mat))
  if (length(dead)) {
    keep <- setdiff(seq_len(nrow(temp_mat)), dead)
    temp_mat <- temp_mat[keep, ]; da_age_c <- da_age_c[keep, ]
  }

  n  <- nrow(da_age_c)
  mu <- rnorm(3, mean = 0, sd = mu_sd)
  tf <- data.table(DAUID = da_age_c$ALT_GEO_CODE, da_idx = 1:n,
                   F1 = rnorm(n, mu[1], 1),
                   F2 = rnorm(n, mu[2], 1),
                   F3 = rnorm(n, mu[3], 1))
  Zc <- as.matrix(tf[, .(F1,F2,F3)]) %*% t(L) + matrix(rnorm(n*17, sd = 0.3), nrow = n)
  colnames(Zc) <- paste0("vuln", sprintf("%02d", 1:17))
  da_age_c[, da_idx := 1:n]
  list(temp_mat = temp_mat, truth_factors = tf, Z = Zc, da_age = da_age_c,
       n_da = n, mu = mu)
}

## 9.6 Simulate mortality - Montréal

Fire DGP on MTL substrate: U-shaped log-RR around each DA's 80-th percentile MMT, modulated by latent factors, lag-convolved, Poisson-drawn. Same `simulate_counts` as §5.5, pointed at MTL.

In [ ]:
mmt_mtl <- apply(mtl$temp_mat, 1, function(x) quantile(x, 0.80, na.rm = TRUE))

mtl_da_long <- CJ(da_idx = 1:mtl$n_da, date = study_dates, age_band = names(annual_rates))
pop_long <- melt(mtl$da_age[, .(da_idx, age_0_64, age_65_74, age_75_84, age_85p)],
                 id.vars = "da_idx", variable.name = "age_band", value.name = "pop")
pop_long[, age_band := as.character(age_band)]
mtl_da_long <- pop_long[mtl_da_long, on = c("da_idx", "age_band")]
mtl_da_long[, annual_rate := annual_rates[age_band]]
mtl_da_long[, lambda0 := pop * annual_rate / 1000 / 365]

mtl_sim <- simulate_counts(mtl$temp_mat, mtl_da_long, mtl$truth_factors, mmt_mtl, seed = 42)

cat("mtl_sim rows:", nrow(mtl_sim), " NA pop:", sum(is.na(mtl_da_long$pop)),
    " total deaths:", sum(mtl_sim$n_deaths), "\n")
print(mtl_sim[, .(deaths = sum(n_deaths)), by = age_band])

Result: 19,902,240 rows (6504 x 765 x 4), zero NA pop. 142,934 deaths over 5 warm-seasons. Deaths climb by age band: 6,776 -> 32,388 -> 45,847 -> 57,923 (0-64 -> 85+). 99.3% of cells zero

## 9.6b run_city — one call, CSV to reduced curve

Chains the per-city path into one function: `read_daymet` → `build_city_sim_substrate_v2` → per-DA 80th-pct MMT → da_long → `simulate_counts` → 150-DA sliver → `build_crossbasis` → `fit_stage1` → `reduce_fit`. Returns `red`, `res`, a one-row `diag`, `Z`, and `da_ids`. The sim is dropped as soon as the sliver is cut, the substrate after the reduce. Memory sat at 1.3 GB after Toronto's 23.5M-row sim.

In [ ]:
run_city <- function(csv_name, da_age_city, cma_label, seed_offset,
                     L, age = "age_75_84", n_sliver = 150, verbose = TRUE) {

  if (verbose) cat(sprintf("\n===== %s =====\n", cma_label))

  dm  <- read_daymet(file.path(DRIVE, csv_name))
  sub <- build_city_sim_substrate_v2(dm, da_age_city, L, seed = 42 + seed_offset)
  rm(dm); gc(verbose = FALSE)

  stopifnot(all(rownames(sub$temp_mat) == sub$truth_factors$DAUID))
  if (verbose) cat(sprintf("  n_da %d  mu [%s]  temp NA %d\n",
                           sub$n_da, paste(round(sub$mu, 3), collapse=", "),
                           sum(is.na(sub$temp_mat))))

  mmt <- apply(sub$temp_mat, 1, function(x) quantile(x, 0.80, na.rm = TRUE))

  dal <- CJ(da_idx = 1:sub$n_da, date = study_dates, age_band = names(annual_rates))
  pl  <- melt(sub$da_age[, .(da_idx, age_0_64, age_65_74, age_75_84, age_85p)],
              id.vars = "da_idx", variable.name = "age_band", value.name = "pop")
  pl[, age_band := as.character(age_band)]
  dal <- pl[dal, on = c("da_idx","age_band")]
  dal[, annual_rate := annual_rates[age_band]]
  dal[, lambda0 := pop * annual_rate / 1000 / 365]
  stopifnot(sum(is.na(dal$pop)) == 0)

  sim <- simulate_counts(sub$temp_mat, dal, sub$truth_factors, mmt, seed = 42)
  rm(dal, pl); gc(verbose = FALSE)
  tot_deaths <- sum(sim$n_deaths)
  if (verbose) cat(sprintf("  deaths %s  lambda NA/Inf %d/%d  pct zero %.2f\n",
                           format(tot_deaths, big.mark=","),
                           sum(is.na(sim$lambda)), sum(is.infinite(sim$lambda)),
                           100*mean(sim$n_deaths == 0)))

  set.seed(42)
  idx <- sample(unique(sim$da_idx), n_sliver)
  sl  <- sim[da_idx %in% idx & age_band == age]
  sl[, DA_id := da_idx]
  setorder(sl, da_idx, date)
  rm(sim); gc(verbose = FALSE)

  tl <- data.table(da_idx = rep(idx, each = length(study_dates)),
                   date   = rep(study_dates, times = n_sliver),
                   temp_C = as.vector(t(sub$temp_mat[idx, ])))
  sl <- tl[sl, on = c("da_idx","date")]
  stopifnot(nrow(sl) == n_sliver * length(study_dates), sum(is.na(sl$temp_C)) == 0)

  cb <- build_crossbasis(sl$temp_C, lag_max = 21)
  dt <- data.table(n_deaths = sl$n_deaths, DA_id = sl$DA_id, date = sl$date)
  stopifnot(nrow(dt) == nrow(cb))

  res <- fit_stage1(dt, cb, cma_label, age, verbose = verbose)
  if (!identical(res$status, "ok")) {
    if (verbose) cat("  stage 1 failed\n")
    return(list(cma = cma_label, status = "failed", res = res))
  }

  ref  <- median(sl$temp_C)
  red  <- reduce_fit(res, ref)

  diag <- data.table(cma = cma_label, n_da = sub$n_da,
                     mu1 = sub$mu[1], mu2 = sub$mu[2], mu3 = sub$mu[3],
                     deaths = tot_deaths, sliver_deaths = sum(sl$n_deaths),
                     ref_temp = ref, winner = res$winner_variant,
                     qaic_A = res$qaic_A, qaic_B = res$qaic_B)

  Zc <- sub$Z; da_ids <- sub$truth_factors$DAUID
  rm(sub, sl, tl, dt); gc(verbose = FALSE)

  list(cma = cma_label, status = "ok", red = red, res = res,
       diag = diag, Z = Zc, da_ids = da_ids)
}

tor_out <- run_city("toronto_daymet_2015_2019.csv",   da_age,
                    "Toronto",   seed_offset = 35,  L = L)
mtl_out <- run_city("montreal_daymet_2015_2019.csv",  cma_age_data_mtlvan$montreal,
                    "Montreal",  seed_offset = 24,  L = L)
van_out <- run_city("vancouver_daymet_2015_2019.csv", cma_age_data_mtlvan$vancouver,
                    "Vancouver", seed_offset = 59,  L = L)

Result: three cities, all Variant A, all V_star positive-definite.

| CMA | n_da | μ | deaths | sliver deaths | ref_temp | qAIC A |
|---|---|---|---|---|---|---|
| Toronto | 7682 | [-0.330, 0.655, 0.384] | 610,216 | 2,419 | 19.38 | 27,047.0 |
| Montréal | 6504 | [1.394, 0.130, 0.251] | 203,988 | 1,524 | 19.01 | 27,967.1 |
| Vancouver | 3573 | [-0.196, 0.331, -0.405] | 56,157 | 649 | 17.05 | 27,442.6 |

Strata 3,750 (A) / 26,250 (B) every city, fixed by the sliver at 150 DAs × 5 years × 5 months, × 7 DOW under B. A wins across all three; MTL runs 0.06 deaths per stratum under B against 0.41 under A.

Multiplier against v1 tracks μ₂ through `exp(0.3·F2·cold)`: Toronto 0.655 → 3.2×, MTL 0.130 → 1.43×, VAN 0.331 → 1.02×. Not monotone in μ₂ — VAN is maritime, fewer days below its own MMT.

V_star diagonals scale inversely with sliver deaths: MTL 4–11 on 1,524, VAN 16–37 on 649. VAN contributes least to the pool.

## 9.7 Scaling to N_CMA=6—Ottawa, Calgary, Québec City geography

Ports §9.1/§9.3 to three more CMAs. Requires `da_all` + `dgrf` live (§1); reload §1.1/§1.2 if absent. DGUIDs: Ottawa 2021S0503505, Calgary 2021S0503825, Québec 2021S0503421. Per CMA: DGRF filter → dedup DA → join national `da_all` → polygons → shapefile bundle to Drive (EE asset) + `st_bbox` in WGS84. Then verify each zip has 4 entries (shp/shx/dbf/prj)

In [ ]:
cat("dgrf exists:", exists("dgrf"), " da_all exists:", exists("da_all"), "\n")

if (exists("dgrf") && exists("da_all")) {
  new_dguids <- c(Ottawa   = "2021S0503505",
                  Calgary  = "2021S0503825",
                  Quebec   = "2021S0503421")

  new_da_lists <- lapply(new_dguids, function(dg) {
    db_rows <- dgrf[CMADGUID_RMRIDUGD == dg]
    dauids  <- substr(unique(db_rows$DADGUID_ADIDUGD), 10, 17)
    list(dauids = dauids)
  })

  new_polys <- lapply(names(new_da_lists), function(nm) {
    dauids <- new_da_lists[[nm]]$dauids
    poly   <- da_all[da_all$DAUID %in% dauids, ]
    list(poly = poly, n = nrow(poly),
         missing = length(setdiff(dauids, poly$DAUID)),
         prefixes = paste(sort(unique(substr(dauids, 1, 2))), collapse=","))
  })
  names(new_polys) <- names(new_da_lists)

  for (nm in names(new_polys)) {
    p <- new_polys[[nm]]
    poly <- p$poly; poly$DAUID <- as.character(poly$DAUID)
    out_dir <- sprintf("/content/%s_da_shp", tolower(nm))
    dir.create(out_dir, showWarnings = FALSE)
    st_write(poly["DAUID"], file.path(out_dir, sprintf("%s_da.shp", tolower(nm))),
             delete_dsn = TRUE, quiet = TRUE)
    zp <- sprintf("%s/%s_da_shp.zip", DRIVE, tolower(nm))
    if (file.exists(zp)) file.remove(zp)
    zip(zipfile = zp, files = list.files(out_dir, full.names = TRUE), flags = "-j")

    poly_wgs <- sf::st_transform(poly, 4326); bb <- sf::st_bbox(poly_wgs)  # bbox from polygons in WGS84 - hand-typed box clips edge DAs
    cat(sprintf("%-8s DAs:%5d  missing:%d  prefix:%-6s  bbox:[%.2f,%.2f,%.2f,%.2f]\n",
                nm, p$n, p$missing, p$prefixes,
                bb["xmin"]-0.05, bb["ymin"]-0.05, bb["xmax"]+0.05, bb["ymax"]+0.05))
  }
} else {
  cat("missing dgrf and/or da_all — reload §1.1 (national shapefile) + §1.2 (DGRF) first.\n")
}

for (nm in c("ottawa","calgary","quebec")) {
  d  <- sprintf("/content/%s_da_shp", nm)
  zp <- sprintf("%s/%s_da_shp.zip", DRIVE, nm)
  files   <- if (dir.exists(d)) list.files(d) else character(0)
  entries <- if (file.exists(zp)) nrow(unzip(zp, list = TRUE)) else NA
  cat(sprintf("%-8s dir_files:%d [%s]  zip_entries:%s\n",
              nm, length(files), paste(files, collapse=" "), entries))
}

Result: Ottawa 2045 DAs (prefix 24,35—bi-provincial), Calgary 1898 (48), Québec 1317 (24). Missing 0 all three. Boxes: OTT [−76.69, 44.80, −75.03, 46.04], CAL [−114.78, 50.74, −113.33, 51.54], QC [−71.86, 46.48, −70.66, 47.36]. Zip entries 4/4/4 verified and all bundles intact.

Note on Ottawa prefix 24,35: Ottawa–Gatineau straddles the river, one CMA across two provinces. Filtering on CMA DGUID catches both banks—a province filter would have halved the city.

## 9.7b Population — Ottawa + Québec City

Ontario and Quebec profiles downloaded, DGRF filtered to Ottawa and Québec City, each city's DAs grepped out and aggregated to four age bands. Ottawa is bi-provincial (prefixes 24 and 35), so it greps both profiles and stacks the parts before aggregating. The Quebec profile is downloaded once and read twice. Both cities then run through `run_city`, and the μ table is checked for duplicates.

DA-level Census Profile files are per-region: Atlantic, Quebec, Ontario, Prairies, BC, Territories. Read the GEONO off the download page's geography dropdown.

In [ ]:
options(timeout = 3600)
base_url <- "https://www12.statcan.gc.ca/census-recensement/2021/dp-pd/prof/details/download-telecharger/comp/GetFile.cfm?Lang=E&FILETYPE=CSV&GEONO="
geono    <- c(ontario = "006_Ontario", quebec = "006_Quebec")
prov_sfx <- c(ontario = "Ontario",     quebec = "Quebec")

for (p in names(geono)) {
  dest <- sprintf("/content/statcan/profile_%s.zip", p)
  if (!file.exists(dest)) system(sprintf('wget -q -O "%s" "%s%s"', dest, base_url, geono[p]))
  ex <- sprintf("/content/statcan/profile_%s_extracted", p)
  if (!dir.exists(ex)) { dir.create(ex); unzip(dest, exdir = ex) }
}

new_dguids <- c(Ottawa = "2021S0503505", Quebec = "2021S0503421")
new_da_lists <- lapply(new_dguids, function(dg) {
  substr(unique(dgrf[CMADGUID_RMRIDUGD == dg]$DADGUID_ADIDUGD), 10, 17)
})

age_ids  <- c(10,11,12,14,15,16,17,18,19,20,21,22,23, 25,26, 27,28, 29)
band_map <- data.table(CHARACTERISTIC_ID = age_ids,
  band = c(rep("age_0_64",13), rep("age_65_74",2), rep("age_75_84",2), rep("age_85p",1)))

city_provs <- list(Ottawa = c("ontario","quebec"), Quebec = "quebec")
new_age <- list()

for (city in names(city_provs)) {
  dauids <- new_da_lists[[city]]
  pat <- sprintf("/content/statcan/%s_patterns.txt", tolower(city))
  writeLines(paste0('"', dauids, '"'), pat)

  parts <- list()
  for (p in city_provs[[city]]) {
    src <- sprintf("/content/statcan/profile_%s_extracted/98-401-X2021006_English_CSV_data_%s.csv",
                   p, prov_sfx[p])
    out <- sprintf("/content/statcan/profile_%s_%s.csv", tolower(city), p)
    if (!file.exists(out)) {
      system(sprintf('head -1 "%s" > "%s"', src, out))
      system(sprintf('grep -F -f "%s" "%s" >> "%s"', pat, src, out))
    }
    parts[[p]] <- fread(out, select = c("ALT_GEO_CODE","CHARACTERISTIC_ID","C1_COUNT_TOTAL"))
  }
  prof <- rbindlist(parts)
  prof <- merge(prof[CHARACTERISTIC_ID %in% age_ids], band_map, by = "CHARACTERISTIC_ID")
  prof[is.na(C1_COUNT_TOTAL), C1_COUNT_TOTAL := 0]

  dw <- dcast(prof[, .(pop = sum(C1_COUNT_TOTAL)), by = .(ALT_GEO_CODE, band)],
              ALT_GEO_CODE ~ band, value.var = "pop")
  dw[, total := age_0_64 + age_65_74 + age_75_84 + age_85p]
  dw[, ALT_GEO_CODE := as.character(ALT_GEO_CODE)]
  new_age[[city]] <- dw

  cat(sprintf("%-8s DAs: %5d / %5d  pop: %s\n", city, nrow(dw), length(dauids),
              format(sum(dw$total), big.mark=",")))
}

ott_out <- run_city("ottawa_daymet_2015_2019.csv", new_age$Ottawa,
                    "Ottawa", seed_offset = 63,  L = L)
qc_out  <- run_city("quebec_daymet_2015_2019.csv", new_age$Quebec,
                    "Quebec", seed_offset = 421, L = L)

mu_all <- rbindlist(lapply(list(tor_out, mtl_out, van_out, ott_out, qc_out),
                           function(o) o$diag[, .(cma, mu1, mu2, mu3)]))
print(mu_all)
cat("duplicate mu rows:", sum(duplicated(mu_all[, .(mu1, mu2, mu3)])), " (must be 0)\n")

Result: Ottawa 2,045 DAs / 1,487,965, band shares 83.0 / 9.9 / 5.0 / 2.0. Québec City 1,317 DAs / 839,300, band shares 78.4 / 12.3 / 6.8 / 2.5. Both match published CMA totals.

Through `run_city`: Ottawa n_da 2,042, μ [-0.776, -0.280, 0.080], 51,476 deaths, 1,247 sliver deaths, ref_temp 18.58, qAIC A 28,992.7 / B 66,496.5. Québec n_da 1,310, μ [-0.077, -1.059, 0.200], 26,977 deaths, 982 sliver deaths, ref_temp 16.79, qAIC A 28,083.3 / B 71,150.7. Both Variant A, both V_star positive-definite. μ duplicate check 0 across the five cities.

Québec City is older than Ottawa — 78.4% under 65 against 83.0, and 6.8% in the 75–84 band against 5.0 — so proportionally more population at risk. Ottawa's 2,042 DAs give 1,247 sliver deaths against Vancouver's 3,573 giving 649; the sliver is 150 DAs regardless of city size. Québec's 16.79 is the lowest ref_temp of the five.

## 9.8 EE temperature export—Ottawa/Calgary/Québec (Python)

Same as §9.4, pointed at the three new assets + boxes. Row counts on Drive: ottawa 1,564,426 / calgary 1,451,971 / quebec 1,007,506 — DA × 765 + 1 header at 2,045 / 1,898 / 1,317 DAs.

In [ ]:
import ee
ee.Initialize(project='gen-lang-client-0569405164')

city_boxes = {
    "ottawa_da":  ee.Geometry.Rectangle([-76.69, 44.80, -75.03, 46.04]),
    "calgary_da": ee.Geometry.Rectangle([-114.78, 50.74, -113.33, 51.54]),
    "quebec_da":  ee.Geometry.Rectangle([-71.86, 46.48, -70.66, 47.36]),
}
prefix = {"ottawa_da": "ottawa", "calgary_da": "calgary", "quebec_da": "quebec"}

for city, bbox in city_boxes.items():
    da_fc = ee.FeatureCollection(f'projects/gen-lang-client-0569405164/assets/{city}')
    daymet = (ee.ImageCollection('NASA/ORNL/DAYMET_V4')
              .filterDate('2015-05-01', '2019-10-01')
              .filterBounds(bbox).select(['tmax', 'tmin']))
    def keep_warm(img):
        m = ee.Date(img.get('system:time_start')).get('month')
        return img.set('keep', m.gte(5).And(m.lte(9)))
    warm = daymet.map(keep_warm).filter(ee.Filter.eq('keep', 1))
    def reduce_one_day(img):
        d = ee.Date(img.get('system:time_start')).format('YYYY-MM-dd')
        return img.reduceRegions(collection=da_fc, reducer=ee.Reducer.mean(),
                                 scale=1000, tileScale=4).map(lambda f: f.set('date', d))  # server-side reduce onto DA polygons - wrong/empty asset = empty CSVs
    out = warm.map(reduce_one_day).flatten()
    ee.batch.Export.table.toDrive(
        collection=out, description=f'{prefix[city]}_daymet_2015_2019',
        folder='thesis/dlnm-pilot', fileNamePrefix=f'{prefix[city]}_daymet_2015_2019',
        fileFormat='CSV', selectors=['DAUID', 'date', 'tmax', 'tmin']).start()
    print(f'{city}: task started')

print('all 3 export tasks submitted')

# 10. Axis reconciliation + per-DA downscale

Stage 2's coefficients were estimated on prcomp(Z_means, 5×17); fit_da_pca runs prcomp on the 7,682×17. Different rotation, different scaling. This section rules which basis the DA scores live on, then runs §8.2's per-DA form through it. Requires the §0 restore and the v2 Toronto substrate — §10.0 regenerates it; §5's seed-42 objects are the v1 draw and do not reproduce these results.


## 10.0 Toronto v2 substrate

build_city_sim_substrate_v2 at seed 42+35, the draw the banked results stand on. Keep truth_factors and temp_mat, release the rest. μ must read −0.33 / 0.655 / 0.384 — any other value is the wrong draw and every downstream number is silently wrong.

In [ ]:
dm  <- read_daymet(file.path(DRIVE, "toronto_daymet_2015_2019.csv"))
sub <- build_city_sim_substrate_v2(dm, da_age, L, seed = 42 + 35)
rm(dm); invisible(gc(verbose = FALSE))
truth_factors <- sub$truth_factors
temp_mat      <- sub$temp_mat
cat("mu:", paste(round(sub$mu, 3), collapse = " "), "\n")
rm(sub); invisible(gc(verbose = FALSE))
cat("n_da:", nrow(truth_factors), "\n")
cat("aligned:", all(rownames(temp_mat) == truth_factors$DAUID), "\n")

## 10.1 The two rotations side by side

Refit pca_cma exactly as §7.5 ran it, assert its scores against the banked cma_predictors — bit-identity proves this is the basis the 20 coefficients were estimated on.

In [ ]:
Z_tor <- Z_list$Toronto
Z_means <- t(sapply(Z_list, colMeans))
stopifnot(identical(colnames(Z_tor), colnames(Z_means)))

pca_cma <- prcomp(Z_means, scale. = TRUE)
cat("sdev:", paste(round(pca_cma$sdev, 4), collapse = "  "), "\n")

refit_scores <- pca_cma$x[, 1:3]
banked <- as.matrix(cma_predictors[match(rownames(refit_scores), cma_predictors$CMA),
                                   c("PC1","PC2","PC3")])
cat("max |refit - banked|:", format(max(abs(refit_scores - banked)), digits = 3), "\n")

Result: sdev 2.7075 / 2.3886 / 1.9905 / 0.0449 / 0 — five cities span barely three dimensions; PC1–3 sits at the edge of what the data carries. max |refit − banked| = 0, exact: this is Stage 2's basis.

## 10.2 Correlations and the ruling

Toronto's Z through pca_cma via predict.prcomp against fit_da_pca's own rotation, both against F1/F2/F3.

In [ ]:
stopifnot(all(rownames(Z_tor) == truth_factors$DAUID))

proj_scores <- predict(pca_cma, newdata = Z_tor)[, 1:3]
da_out      <- fit_da_pca(Z_tor, truth_factors$DAUID)
da_scores   <- as.matrix(da_out$scores[, .(PC1, PC2, PC3)])
Fmat        <- as.matrix(truth_factors[, .(F1, F2, F3)])

print(round(cor(proj_scores, Fmat), 3))
print(round(cor(da_scores, Fmat), 3))
print(round(cor(proj_scores, da_scores), 3))

Result: cross-rotation best pairs 0.839 / 0.780 / 0.949 with the axes permuted (proj PC2 ↔ da PC3, proj PC3 ↔ da PC2) and off-axis leakage to 0.617. High but scrambled — both rotations see the same factor structure, weighted differently by 5 averaged points versus 7,682 raw ones.

Ruling (superseded 2026-08-12, §11 — final form is bottom-up, DA scores native): Stage 3 used the pca_cma projection. The rotations are permuted and mixed, so the axis choice is real, and the projection is the basis the 20 coefficients were estimated on. Note: da rotation shows F1 weak at 0.62, not F3 at 0.49 — scores versus loadings, parked for Criterion 3.

## 10.3 Per-DA θ

Coefficient name order asserted before the reshape trusts it.

In [ ]:
cf <- coef(stage2_k5)
stopifnot(identical(names(cf)[1:6],
  c("theta1.(Intercept)","theta2.(Intercept)","theta3.(Intercept)",
    "theta4.(Intercept)","theta5.(Intercept)","theta1.PC1")))

theta_da <- predict_da_theta(stage2_k5, proj_scores, truth_factors$DAUID)

cat("dims:", paste(dim(theta_da), collapse = " x "), "\n")
cat("theta sd:", paste(round(sapply(theta_da[, 3:7], sd), 3), collapse = "  "), "\n")
cat("distinct theta5:", uniqueN(theta_da$theta5), "\n")

Result: 7,682 × 7, distinct theta5 = 7,682 — the flat map. θ SD 5.011 / 3.159 / 3.822 / 2.549 / 7.222, θ₅ widest at 1.4× θ₁ (slope math, not the several-times guess). Projected score SDs ≈ 4.9–5.5 against city sdev 2.0–2.7: the DA cloud spreads ~2× the fit range, stretched not exploded. Medians sit off the intercepts because Toronto is not at PC zero.

## 10.4 RR at p99 and the score

Basis row at 27.19 minus row at cen, each 5-vector through, exp. Then the map scored against exp(0.4·F1 + 0.2·F3) on the log scale.

In [ ]:
av  <- attr(res5$Toronto$cb_template, "argvar")
cen <- red5$Toronto$cen
ob  <- onebasis(c(cen, 27.19), fun = av$fun, degree = av$degree, knots = av$knots,
                Boundary.knots = av$Boundary.knots)
bdiff  <- ob[2, ] - ob[1, ]
log_rr <- as.matrix(theta_da[, 3:7]) %*% bdiff
rr     <- exp(log_rr)

cat("rr sd:", round(sd(rr), 4), " median:", round(median(rr), 3), "\n")
cat("rr range:", paste(round(range(rr), 3), collapse = "  "),
    " quartiles:", paste(round(quantile(rr, c(.25, .75)), 3), collapse = "  "), "\n")

truth_v <- exp(0.4 * truth_factors$F1 + 0.2 * truth_factors$F3)
cat("log sd recovered vs truth:", round(sd(log_rr), 3), round(sd(log(truth_v)), 3), "\n")
cat("pearson (log):", round(cor(log_rr, log(truth_v)), 3),
    " spearman:", round(cor(log_rr, truth_v, method = "spearman"), 3), "\n")
cat("vs F1:", round(cor(log_rr, truth_factors$F1), 3),
    " F2:", round(cor(log_rr, truth_factors$F2), 3),
    " F3:", round(cor(log_rr, truth_factors$F3), 3), "\n")

Result: RR sd 13.2372, median 2.975 (flat map 3.176), range 0.017–294.632, quartiles 1.241 / 7.04. Pearson −0.922, Spearman −0.912: the ordering is near-perfect and globally inverted. Log SD 1.299 against truth 0.445 — amplitude 2.9× on the log scale. Per-factor: F1 −0.687, F3 −0.686, F2 control −0.127. The flip is global, both truth factors inverted together; F1 and F3 contribute equally instead of echoing the 0.4/0.2 weights — the rotation re-mixed the proportions. Per the branch rule the amplitude is recorded, not tuned: right geography, inverted sign, tripled volume. Sign hunt: §11 rules the projection seam out — the flip survives the bottom-up rotation on native scores at −0.946. Diagnosis at §11.5.

## 10.5 First per-DA MMT

compute_da_mmt on one DA — validates §10.3's output and reads the inversion in an independent object. Requires temp_mat from §10.0.

In [ ]:
stopifnot(all(rownames(temp_mat) == theta_da$DAUID))
th1  <- as.numeric(theta_da[1, 3:7])
t_da <- sort(temp_mat[1, ])
mmt1 <- compute_da_mmt(th1, res5$Toronto$cb_template, t_da, red5$Toronto$cen)

cat("da:", theta_da$DAUID[1], " range:", paste(round(range(t_da), 2), collapse = "  "),
    " mmt:", round(mmt1, 2), "\n")

Result: DA 35180018, range 3.33–28.11, MMT 25.24 — interior but 3°C off the hot ceiling, against the flat map's 21.68. A flipped θ says risk falls with heat, so the minimum drifts hot: the inversion again, in a second object. The function is fine; the input is upside down.

# 11. Bottom-up rotation — final form, first run

The 08-11 ruling made the final form's PCA bottom-up: rotation fitted at DA level, CMA scores population-weighted up. The pilot had never run it. Running it also deletes the seam the sign hunt pointed at — one rotation, DA scores native, no projection between estimation basis and prediction basis. §11 builds it end to end and scores the map. The flip survives (−0.946), which retires the seam theory and moves the hunt to the outcome side; §11.6 carries the diagnosis.

Inputs: Z5_v2 / ids5_v2 / red5_v2 / stage2_k5_v2 from the 07-22 tarball, population tables from session da_age + mtl/van substrates + new_age_ottqc. The banked mtl_substrate/van_substrate are v1 (list of 5, no mu) — used here for population columns only, never for Z or truth.

## 11.0 Stack the five city Z matrices

Z5_v2 and ids5_v2 are per-city lists, not a stacked matrix. Fixed city order, per-city row/id assertion before the rbind, city label carried for the weighting. DAUIDs are nationally unique, so duplicates must be 0.

In [ ]:
cities <- c("Toronto","Montreal","Vancouver","Ottawa","Quebec")
stopifnot(identical(names(Z_list), cities), identical(names(ids5), cities))
for (cc in cities) stopifnot(nrow(Z_list[[cc]]) == length(ids5[[cc]]),
                             ncol(Z_list[[cc]]) == 17)
Zs       <- do.call(rbind, Z_list[cities])
ids_all  <- unlist(ids5[cities], use.names = FALSE)
city_all <- rep(cities, times = sapply(Z_list[cities], nrow))

cat("Zs:", paste(dim(Zs), collapse=" x "), "| NAs:", sum(is.na(Zs)), "\n")
cat("ids:", length(ids_all), "| dup ids:", sum(duplicated(ids_all)), "\n")
print(table(city_all))

Result: Zs 21,111 × 17, zero NA, zero duplicate ids. Rows: TOR 7,682 / MTL 6,504 / VAN 3,573 / OTT 2,042 / QUE 1,310. Aligned by construction — the per-city assertions run before the rbind, so Z pairs to ids by position from here on.


## 11.1 The rotation

prcomp scaled on the full stack. This call fits the rotation where prediction happens — every θ in §11 descends from it. Loadings kept for the Criterion 3 recompute.

In [ ]:
pca_da <- fit_da_pca(Zs, ids_all)
scores <- pca_da$scores
scores[, city := city_all]

cat("scores:", paste(dim(scores), collapse=" x "), "| NAs:", sum(is.na(scores)), "\n")
cat("var explained:", round(pca_da$var_explained, 3), "\n")
print(scores[, lapply(.SD, mean), by = city, .SDcols = c("PC1","PC2","PC3")])

Result: cum var 86.9%, split 0.321 / 0.289 / 0.259 — three same-sd factors through L split credit evenly; the missing 13% is the sd-0.3 column noise. Scores 21,111 × 5, zero NA. City means separate ~3 units on PC1.

PC1's city order and signs match the banked top-down cma_predictors exactly (MTL < TOR < VAN < OTT < QUE both rotations), so PC1's column orientation never flipped between levels — an early receipt against the seam theory. Spread is tighter than top-down (~3 vs ~6 units): means of scores, not scores of means — the top-down rotation was fitted on the 5 means and stretches them.

## 11.2 Population weights, CMA scores weighted up

Weights are census totals scattered over four sources: session da_age (TOR), the v1 substrates' da_age (MTL/VAN), new_age_ottqc (OTT/QUE). TOR/OTT/QUE tables run long by 12/3/7 — pre-water-drop vintages — so the join is by DAUID via match(), never by position, and unmatched must land at exactly 0.

In [ ]:
pop_all <- rbind(
  da_age[,            .(DAUID = ALT_GEO_CODE, total)],
  mtl$da_age[,        .(DAUID = ALT_GEO_CODE, total)],
  van$da_age[,        .(DAUID = ALT_GEO_CODE, total)],
  new_age$Ottawa[,    .(DAUID = ALT_GEO_CODE, total)],
  new_age$Quebec[,    .(DAUID = ALT_GEO_CODE, total)]
)
m <- match(ids_all, pop_all$DAUID)
cat("unmatched:", sum(is.na(m)), "\n")
scores[, w := pop_all$total[m]]
cat("w range:", range(scores$w), "| NAs:", sum(is.na(scores$w)), "\n")

cma_bu <- scores[, .(PC1 = weighted.mean(PC1, w),
                     PC2 = weighted.mean(PC2, w),
                     PC3 = weighted.mean(PC3, w)), by = city]
print(cma_bu)

Result: all 21,111 matched, weights 15–29,665, zero NA. Weighted means sit within ~0.07 of the unweighted ones, same order throughout — population is independent of the factors by construction, so the tilt has nothing to grab. cma_bu is the bottom-up predictor table.

In [ ]:
stopifnot(all(c("theta_star","V_star") %in% names(red5$Toronto)))
theta_bu <- t(sapply(red5[cities], function(r) r$theta_star))
V_bu     <- lapply(red5[cities], function(r) r$V_star)
setkey(cma_bu, NULL); cma_bu <- cma_bu[match(cities, city)]
stopifnot(identical(cma_bu$city, cities), nrow(theta_bu) == 5)

stage2_bu <- mixmeta(theta_bu ~ PC1 + PC2 + PC3, S = V_bu,
                     data = cma_bu, method = "reml")

cat("converged:", stage2_bu$converged, "\n")
cat("coefs:", length(coef(stage2_bu)), "| df.resid:", stage2_bu$df.residual, "\n")
print(round(matrix(coef(stage2_bu), nrow = 5), 3))

Result: converged, 20 coefficients, df.residual −10 — the K=5 identity, unchanged by the rotation. Coefficient names run b1..b5 within each predictor block (outcome-fastest), the order matrix(cf, nrow = 5) assumes; §11.5 proves the reshape against fitted(). PC columns nonzero and mixed-sign — the scores do work in the pool.

## 11.4 Native-score θ and the matched-vintage score

Toronto's DA scores from §11.1 straight into predict_da_theta — no projection exists to carry a flipped axis. Contrast weights from the banked Stage 1 argvar with Boundary.knots passed explicitly. Truth regenerated at seed 77 (§10.0 draw) — the session RData's truth_factors is the v1 seed-42 imposter and scores 0.00 against any v2 object.

Two rulers broke before this one held, both recorded as reflexes: (1) onebasis on two points defaults boundary knots to the range of those two points — half the basis is a different basis; sd inflated to 5.66, cor 0.00. (2) big SD with zero correlation is a wrong ruler, not a flipped or dead map — a flip preserves |cor|, zero means the vectors don't know each other, which is what mismatched vintages print.

In [ ]:
tor_sc <- as.matrix(scores[city == "Toronto", .(PC1, PC2, PC3)])
tor_id <- scores[city == "Toronto", DAUID]
stopifnot(identical(tor_id, ids5$Toronto))
th_bu <- predict_da_theta(stage2_bu, tor_sc, tor_id)
cat("distinct rows:", nrow(unique(th_bu[, .(theta1,theta2,theta3,theta4,theta5)])), "\n")

red_tor <- readRDS("/content/saves_eod/red_tor.rds")
av  <- attr(red_tor$cb_template, "argvar")
cen <- red5$Toronto$cen
b2  <- onebasis(c(27.19, cen), fun = av$fun, degree = av$degree,
                knots = av$knots, Boundary.knots = av$Boundary.knots)
wts2   <- b2[1, ] - b2[2, ]
th_mat <- as.matrix(th_bu[, .(theta1,theta2,theta3,theta4,theta5)])
logrr2 <- as.vector(th_mat %*% wts2)

tor_temp <- read_daymet(file.path(DRIVE, "toronto_daymet_2015_2019.csv"))
tor_sub  <- build_city_sim_substrate_v2(tor_temp, da_age, L, seed = 42 + 35)
cat("n_da:", tor_sub$n_da, "| mu:", round(tor_sub$mu, 3), "\n")
cat("Z identical to Z5:", isTRUE(all.equal(tor_sub$Z, Z_list$Toronto, check.attributes = FALSE)),
    "| ids match:", identical(tor_sub$truth_factors$DAUID, tor_id), "\n")

truth2 <- with(tor_sub$truth_factors, 0.4*F1 + 0.2*F3)
cat("pearson:", round(cor(logrr2, truth2), 3),
    "| spearman:", round(cor(logrr2, truth2, method = "spearman"), 3), "\n")
cat("truth sd:", round(sd(truth2), 3), "| logrr sd:", round(sd(logrr2), 3), "\n")

Result: 7,682 distinct θ rows. Regeneration fingerprints all hit — n_da 7,682, μ −0.33 / 0.655 / 0.384, Z bit-identical to the stacked Z5, ids in score order. Score: pearson −0.946, spearman −0.940, truth sd 0.445, log sd 1.226 — amplitude 2.76×.

The flip survives a seamless build. Same signature as §10.4 (−0.922, 2.9×) under a different rotation: the seam theory is dead, and whatever inverts the map is shared by both runs and sits downstream of the rotation choice.


## 11.5 Diagnosis — three suspects cleared, one isolated

Reshape on trial first: predict at the five city-mean score rows and diff against fitted() — the model's own fitted values are ground truth no reshape can fake. Then the whole downstream collapses to one 3-vector: a = wts2 · B[,2:4] is the only thing the map applies to any DA's scores, and its alignment with cv = cor(tor_sc, truth2) is the sign of the map in one number.

In [ ]:
cma_hat <- predict_da_theta(stage2_bu,
                            as.matrix(cma_bu[, .(PC1, PC2, PC3)]),
                            cma_bu$city)
print(round(as.matrix(cma_hat[, .(theta1,theta2,theta3,theta4,theta5)]) -
            fitted(stage2_bu), 4))

B <- matrix(coef(stage2_bu), nrow = 5)
a <- as.vector(wts2 %*% B[, 2:4])
cv <- cor(tor_sc, truth2)[, 1]
cat("a:", round(a, 3), "\n")
cat("cor(PC, truth):", round(cv, 3), "\n")
cat("alignment a.cv:", round(sum(a * cv), 3), "\n")
cat("check cor(a.s, logrr2):", round(cor(as.vector(tor_sc %*% a), logrr2), 4), "\n")
cat("labels:", sapply(red5[cities], `[[`, "cma"), "\n")

Result: diff table dead zeros — reshape acquitted. a = (0.424, −0.349, −0.343), cv = (−0.557, 0.595, 0.418), alignment −0.588 of a possible ±0.594 (|a||cv| = 0.648 × 0.916): a is antiparallel to the truth direction at 99% of maximum. Compression check exact (cor 1) — the whole map is a·s. red5 internal cma labels match the list names — pairing acquitted.

Predictors acquitted by arithmetic: cities' mean scores projected onto cv track SSOT mean truth (0.4μ1 + 0.2μ3) in the same order both ways — OTT −1.52/−0.294 lowest through MTL +1.75/+0.608 highest. Z is linear in F, city scores carry city truth faithfully. So the reversal lives entirely on the outcome side: the five city θ*'s anti-track mean truth through Stage 2.

Caution banked from the failed reads: a city's θ evaluated on another city's basis is soup — each θ* lives on its own city-quantile knots (OTT read 3.39 log-RR through Toronto's basis, an RR of 30, impossible under a DGP whose modulation caps near e^0.6). And a curve read at quantile(predvar, .99) sits in the extrapolation tail past the last knot — a grid quantile is not a data quantile; QUE read −3.9 there.

## 11.6 The sign branch — each city's curve at its own p90 knot

reduced_obj$basis is a onebasis matrix; knots ride on it as attributes, so the read is attr(basis, "knots")[3] — and the knots carry a quantile label ("Toronto.90%"), so unname() before any lookup by city name, or the lookup returns NA (reflex, 2026-08-21). Own knot on own curve is the only percentile-comparable read: inside the support, same percentile for every city. The committed branch, written before the run: track → θ*-to-curve on trial; anti-track → Stage 1 on trial.

In [ ]:
mus <- rbind(Toronto   = c(-0.330, 0.655, 0.384),
             Montreal  = c( 1.394, 0.130, 0.251),
             Vancouver = c(-0.196, 0.331,-0.405),
             Ottawa    = c(-0.776,-0.280, 0.080),
             Quebec    = c(-0.077,-1.059, 0.200))

knot3 <- sapply(cities, function(cc) unname(attr(red5[[cc]]$reduced_obj$basis, "knots")[3]))
fit_p90 <- sapply(cities, function(cc) {
  r <- red5[[cc]]$reduced_obj
  stopifnot(knot3[cc] > min(r$predvar), knot3[cc] < max(r$predvar))
  r$fit[which.min(abs(r$predvar - knot3[cc]))]
})

tab1 <- data.table(city = cities,
                   mean_truth = round(0.4*mus[cities,1] + 0.2*mus[cities,3], 3),
                   knot3 = round(knot3, 2),
                   fit_p90 = round(fit_p90, 3))
print(tab1)
cat("spearman(truth, fit):", round(cor(tab1$mean_truth, tab1$fit_p90, method = "spearman"), 3), "\n")

Result: Spearman −0.9, anti-track. TOR −0.055/0.261, MTL 0.608/−0.851, VAN −0.159/0.689, OTT −0.294/0.458, QUE 0.009/−0.136. Montréal alone makes the case — highest mean truth, lowest curve, log-RR −0.85 at its own p90 is RR 0.43, protective at heat, impossible under a DGP whose base_log_rr is positive at heat. The reversal is born before pooling. Verdict per the branch: Stage 1 is on trial; the inside move is a per-city simulation read directly against base_log_rr; the choropleth is gated — a map from inverted curves is a receipt of the wrong thing.

## 11.7 Criterion 3 on the bottom-up loadings

Sign-independent, so it runs regardless of the branch. Loadings live at pca_da$pca$rotation — fit_da_pca returns list(pca, scores, var_explained), no top-level $rotation; asking for the absent slot yields junk that cor() recycles into cor(L) against itself, printing a fake exact-1.000 diagonal (reflex, 2026-08-21: exact 1 is a mirror, not a match — PCs are orthogonal, L's columns aren't, exact recovery is impossible; the is.matrix guard under the extraction is what catches it). Scored as best |cor| per true factor: sign-free by abs, permutation-free by row max.

In [ ]:
stopifnot(identical(names(Z_list), cities), identical(names(ids5), cities))
for (cc in cities) stopifnot(nrow(Z_list[[cc]]) == length(ids5[[cc]]), ncol(Z_list[[cc]]) == 17)
Zs      <- do.call(rbind, Z_list[cities])
ids_all <- unlist(ids5[cities], use.names = FALSE)
stopifnot(nrow(Zs) == 21111, sum(duplicated(ids_all)) == 0)

pca_da <- fit_da_pca(Zs, ids_all)

ld <- pca_da$pca$rotation[, 1:3]
stopifnot(is.matrix(ld), is.numeric(ld), dim(ld) == c(17, 3))
cat("ld orthogonal:", round(max(abs(crossprod(ld)[upper.tri(crossprod(ld))])), 6), "\n")

cm <- abs(cor(L, ld))
dimnames(cm) <- list(paste0("F", 1:3), paste0("PC", 1:3))
print(round(cm, 3))
best <- apply(cm, 1, max)
cat("best |cor| per factor:", round(best, 3), "\n")
cat("C3 mean |cor|:", round(mean(best), 3), "| pass >0.7:", mean(best) > 0.7, "\n")

save_to_drive(list(tab1_p90_sign = tab1, loadings_bu = ld, c3_bu = best,
                   pca_da_var = pca_da$var_explained), tag = "2026-08-21")

Result: orthogonality 0, cum var 86.9%. Grid F1→PC2 0.947, F2→PC1 0.801, F3→PC3 0.800, F2 bleeding into PC3 at 0.72. C3 = 0.849, pass, replaces the stale 0.731. Banked to saves_eod_2026-08-21: tab1_p90_sign, loadings_bu, c3_bu, pca_da_var — everything else in §11 regenerates from the 07-22 tarball.